# Webcam Discovery — Location-Agnostic Step-by-Step Debug Notebook with Fail-Fast Preflight and Active Heartbeat for Colab

This notebook tests each stage of the `webcam-discovery` application using real public network calls and real application artifacts. It does **not** use synthetic streams, mock data, simulated runs, or fabricated benchmark output.

This version is intentionally **not bound to California, Caltrans, WZMedia, or any other single source/location**. The user query is the source of truth. The notebook derives a target scope from the query, then audits every artifact against that scope.

It is intended for debugging these failure modes while the run is still active. The heartbeat prints the current stage, recent findings, source/domain counts, cap usage, and query-derived location leakage alerts so a bad run can be interrupted early.

It is intended for debugging these failure modes:

- location leakage, such as a query for one place accepting similarly named places elsewhere
- ambiguous same-name places, such as `Cameron` matching unrelated towns, regions, or countries
- stream URL metadata extraction failures for district, route, direction, road/street, landmark, agency, or place hints
- candidate and stream counts exceeding configured limits
- valid streams being lost before validation or collapsed during dedup/catalog export
- SSL certificate failures in Colab
- missing or incomplete per-stage artifacts

The notebook can optionally use the same Ollama Cloud key used by the application to parse the query target and to interpret difficult stream URLs. Heuristic parsing is always run first; LLM extraction is optional and audit-only.


This version also fails fast before the expensive run when the source registry is empty or the planner LLM is unreachable, because those failures prevent meaningful discovery debugging.


In [1]:
# 01. Runtime configuration — edit this cell for each debug run.

REPO_URL = "https://github.com/dshipley71/webcam-discovery.git"
# Use main or a Codex branch you want to validate.
BRANCH = "main"

# Main natural-language request. This is the ONLY required target input.
# Set this explicitly for each Colab run; the notebook intentionally has no default place.
# Example: QUERY = "Get me public live HLS cameras near <your place or landmark>"
QUERY = ""

# Target-scope audit behavior.
# The notebook derives TARGET_SCOPE from QUERY. There is no default bbox or place.
# Manual overrides are optional and should be query-specific, not application defaults.
TARGET_SCOPE_MODE = "LLM_FROM_QUERY"  # LLM_FROM_QUERY or MANUAL. No heuristic fallback is allowed.
MANUAL_TARGET_SCOPE = {
    "requested_places": [],
    "country_terms": [],
    "admin1_terms": [],
    "admin2_terms": [],
    "locality_terms": [],
    "landmark_terms": [],
    "positive_terms": [],            # query-specific allowed strings/domains/agencies
    "negative_terms": [],            # query-specific known wrong places, e.g. ["Ghana", "Cameroon"]
    "bbox": None,                    # optional query-derived/manual bbox only; never defaulted
}

# Optional LLM audit. These calls are only for notebook diagnostics, not fake app output.
# IMPORTANT: this notebook intentionally uses LLM extraction for target scope and URL metadata.
# It must fail fast rather than falling back to heuristic parsing.
USE_OLLAMA_CLOUD = True
OLLAMA_SECRET_NAME = "OLLAMA_API_KEY"
PLANNER_PROVIDER = "ollama"
PLANNER_MODEL = "gemma4:31b-cloud"
PLANNER_BASE_URL = "https://ollama.com"
OLLAMA_PREFLIGHT_TIMEOUT_SECONDS = 180
# LLM extraction remains mandatory, but cloud calls are retried instead of failing on one transient read timeout.
OLLAMA_EXTRACTION_CONNECT_TIMEOUT_SECONDS = 20
OLLAMA_EXTRACTION_READ_TIMEOUT_SECONDS = 240
OLLAMA_EXTRACTION_MAX_ATTEMPTS = 3
OLLAMA_EXTRACTION_BACKOFF_SECONDS = 8
# Keep extraction on the configured planner model by default; change only if you intentionally want a faster cloud model for JSON extraction.
LLM_EXTRACTION_MODEL = PLANNER_MODEL
FAIL_FAST_ON_LLM_PREFLIGHT_FAILURE = True
FAIL_FAST_ON_ZERO_SOURCE_REGISTRY = True
PLANNER_QUIET_WARNING_SECONDS = 30
ENABLE_LLM_TARGET_SCOPE_AUDIT = True
ENABLE_LLM_URL_METADATA_AUDIT = True
MAX_LLM_URL_METADATA_ROWS = 25

# Debug presets. EXTENDED is useful when the app is under-discovering streams.
LIMIT_PRESET = "EXTENDED"  # QUICK, STANDARD, EXTENDED, CUSTOM

PRESETS = {
    "QUICK": {
        "MAX_FEED_ENDPOINTS": 40,
        "MAX_FEED_RECORDS": 1500,
        "MAX_STREAM_CANDIDATES": 1000,
        "PER_SOURCE_STREAM_CAP": 250,
        "MAX_STREAMS": 200,
        "MAX_SEARCH_QUERIES": 12,
        "MAX_SEARCH_RESULTS_PER_QUERY": 8,
    },
    "STANDARD": {
        "MAX_FEED_ENDPOINTS": 100,
        "MAX_FEED_RECORDS": 3000,
        "MAX_STREAM_CANDIDATES": 2500,
        "PER_SOURCE_STREAM_CAP": 500,
        "MAX_STREAMS": 500,
        "MAX_SEARCH_QUERIES": 25,
        "MAX_SEARCH_RESULTS_PER_QUERY": 10,
    },
    "EXTENDED": {
        "MAX_FEED_ENDPOINTS": 500,
        "MAX_FEED_RECORDS": 15000,
        "MAX_STREAM_CANDIDATES": 10000,
        "PER_SOURCE_STREAM_CAP": 2000,
        "MAX_STREAMS": 2500,
        "MAX_SEARCH_QUERIES": 40,
        "MAX_SEARCH_RESULTS_PER_QUERY": 20,
    },
    "CUSTOM": {
        "MAX_FEED_ENDPOINTS": 100,
        "MAX_FEED_RECORDS": 3000,
        "MAX_STREAM_CANDIDATES": 2500,
        "PER_SOURCE_STREAM_CAP": 500,
        "MAX_STREAMS": 500,
        "MAX_SEARCH_QUERIES": 25,
        "MAX_SEARCH_RESULTS_PER_QUERY": 10,
    },
}

limits = PRESETS[LIMIT_PRESET].copy()
MAX_FEED_ENDPOINTS = limits["MAX_FEED_ENDPOINTS"]
MAX_FEED_RECORDS = limits["MAX_FEED_RECORDS"]
MAX_STREAM_CANDIDATES = limits["MAX_STREAM_CANDIDATES"]
PER_SOURCE_STREAM_CAP = limits["PER_SOURCE_STREAM_CAP"]
MAX_STREAMS = limits["MAX_STREAMS"]
MAX_SEARCH_QUERIES = limits["MAX_SEARCH_QUERIES"]
MAX_SEARCH_RESULTS_PER_QUERY = limits["MAX_SEARCH_RESULTS_PER_QUERY"]

ENABLE_FEED_DISCOVERY = True
PRESERVE_DIRECT_STREAMS = True
ENABLE_VISUAL_ANALYSIS = True
ENABLE_MEMORY = False
ALLOW_INSECURE_SSL_FALLBACK = False

# Validation debug controls. These are developer/debug flags, not production defaults.
DISABLE_FFPROBE_VALIDATION = False
DISABLE_VALIDATION = False

# Source governance should remain enabled by default.
# Only set True when intentionally testing search-only behavior.
IGNORE_SOURCES_MD = False

# LLM scope-enforcement validation. This validates the app's real LLM scope artifacts.
ENABLE_LLM_SCOPE_ENFORCEMENT_VALIDATION = True
RUN_INSUFFICIENT_SCOPE_VALIDATION = True
INSUFFICIENT_SCOPE_QUERY = "Find public HLS cameras"

# Optional real multi-scope validation. Keep False by default if runtime is a concern.
RUN_MULTI_SCOPE_VALIDATION = False
MULTI_SCOPE_QUERY = "Get me public HLS cameras from London, England and Sydney, Australia"


# Active heartbeat / live-run guardrails.
# These settings do not change application behavior; they monitor the running subprocess.
HEARTBEAT_SECONDS = 10
HEARTBEAT_RECENT_LINES = 12
HEARTBEAT_SHOW_DOMAIN_TOP_N = 12
HEARTBEAT_STATUS_JSON = True

# Location-leakage monitoring uses the query-derived TARGET_SCOPE.
# Example: for "Cameron, Texas", the LLM/override can include wrong same-name terms
# such as "Cameroon", "Ghana", or "Africa" as ambiguous/off-target terms.
MONITOR_LOCATION_LEAKAGE = True
EXTRA_LOCATION_LEAKAGE_TERMS = []  # optional query-specific terms, e.g. ["Ghana", "Cameroon", "Africa"]
AUTO_STOP_ON_LOCATION_LEAKAGE = False  # set True when you want the notebook to terminate the run automatically
LOCATION_LEAKAGE_STOP_HITS = 3
LOCATION_LEAKAGE_STOP_GRACE_SECONDS = 45

# Optional hard runtime guard. Set 0 to disable.
MAX_RUNTIME_MINUTES = 0

# Optional manual stop-file guard. In another Colab terminal/file-browser action, create this file
# to request the running subprocess terminate cleanly. Cell interruption also works.
STOP_FILE = "/content/STOP_WEBCAM_DISCOVERY"

# Output behavior.
RUN_LABEL = "location_agnostic_step_debug"
ARTIFACT_ROOT = "/content/webcam_discovery_debug_outputs"
ASSERT_ON_WARNINGS = False  # Set True when using this notebook as a strict acceptance gate.

if not QUERY or "<" in QUERY or ">" in QUERY:
    raise ValueError("Set QUERY to a real location-aware camera discovery request before running.")

if not QUERY.strip():
    raise RuntimeError("Set QUERY in the runtime configuration cell before running discovery. The notebook has no default target scope.")
print("QUERY:", QUERY)
print("BRANCH:", BRANCH)
print("TARGET_SCOPE_MODE:", TARGET_SCOPE_MODE)
print("LIMIT_PRESET:", LIMIT_PRESET)
print("Effective limits:", limits)


QUERY: Get me public live traffic cameras from Cameron, Texas
BRANCH: codex/add-llm-based-scope-enforcement-agent
TARGET_SCOPE_MODE: LLM_FROM_QUERY
LIMIT_PRESET: EXTENDED
Effective limits: {'MAX_FEED_ENDPOINTS': 500, 'MAX_FEED_RECORDS': 15000, 'MAX_STREAM_CANDIDATES': 10000, 'PER_SOURCE_STREAM_CAP': 2000, 'MAX_STREAMS': 2500, 'MAX_SEARCH_QUERIES': 40, 'MAX_SEARCH_RESULTS_PER_QUERY': 20}


## 02. Install repository and runtime dependencies

This cell clones or updates the repository, installs `ffmpeg`, installs the package in editable mode, and prints the CLI help. It does not overwrite `SOURCES.md`; if the app has a source-registry problem, the notebook should expose it rather than hide it.


In [2]:
import os, sys, subprocess, pathlib, shutil, json, textwrap, re, time, datetime

repo_dir = pathlib.Path("/content/webcam-discovery")
if repo_dir.exists():
    subprocess.run(["git", "fetch", "--all", "--prune"], cwd=repo_dir, check=False)
    subprocess.run(["git", "checkout", BRANCH], cwd=repo_dir, check=True)
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo_dir, check=False)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo_dir)], check=True)

subprocess.run(["apt-get", "update", "-qq"], check=False)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "ca-certificates"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "certifi", "truststore", "pandas"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=repo_dir, check=True)

print("Repo ready:", repo_dir)
print("Branch:")
subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=repo_dir, check=False)
print("Commit:")
subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=repo_dir, check=False)
print("\nCLI help:")
subprocess.run(["webcam-discovery", "run-agentic", "--help"], cwd=repo_dir, check=False)


# Colab/Jupyter import hardening:
# Editable installs sometimes do not refresh the current kernel import path immediately.
# Always expose the repository src/ directory explicitly before importing webcam_discovery.
os.chdir(repo_dir)
src_dir = repo_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

try:
    import webcam_discovery
    print("Import validation OK:", webcam_discovery.__file__)
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "Failed to import webcam_discovery after clone/install. "
        f"repo_dir={repo_dir}, src_dir={src_dir}, cwd={pathlib.Path.cwd()}, sys.path[:5]={sys.path[:5]}"
    ) from exc


Repo ready: /content/webcam-discovery
Branch:
Commit:

CLI help:
Import validation OK: /content/webcam-discovery/src/webcam_discovery/__init__.py


## 03. Repository policy and source-registry preflight

This step verifies that the repo-level instructions and source registry are present. It also scans for terms that can cause privacy or non-public source problems. The goal is to catch policy drift before discovery starts.


In [3]:
import pathlib, re, pandas as pd, json, os, sys, subprocess, inspect

required_docs = ["AGENTS.md", "SKILLS.md", "SOURCES.md", "README.md"]
rows = []
for name in required_docs:
    p = repo_dir / name
    text = p.read_text(encoding="utf-8", errors="replace") if p.exists() else ""
    rows.append({
        "file": name,
        "exists": p.exists(),
        "bytes": p.stat().st_size if p.exists() else 0,
        "mentions_public_only": "public" in text.lower(),
        "mentions_hls_or_m3u8": ("hls" in text.lower() or ".m3u8" in text.lower()),
        "mentions_no_synthetic_or_mock": any(k in text.lower() for k in ["no synthetic", "mock", "simulated"]),
    })
display(pd.DataFrame(rows))

source_path = repo_dir / "SOURCES.md"
if not source_path.exists():
    raise RuntimeError("SOURCES.md not found. The directory/source preflight cannot continue.")

source_text = source_path.read_text(encoding="utf-8", errors="replace")
blocked_terms = ["insecam", "shodan", "censys", "zoomeye", "fofa"]
print("Blocked/source-risk terms found in SOURCES.md:", {t: source_text.lower().count(t) for t in blocked_terms})

# Import hardening: do not assume editable install state in the active Colab kernel.
# Use the repository package path explicitly before importing application code.
src_dir = repo_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Use the application's own SourcesRegistry parser. This catches the exact failure shown by
# `SourcesRegistry: loaded tiers {1: 0, 2: 0, 3: 0} (0 total sources)` before a long run starts.
try:
    from webcam_discovery.agents.directory_crawler import SourcesRegistry
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "Cannot import webcam_discovery.agents.directory_crawler.SourcesRegistry. "
        "Run the setup/install cell first, or verify that repo_dir points to a cloned webcam-discovery repo. "
        f"repo_dir={repo_dir}, src_dir={src_dir}, cwd={pathlib.Path.cwd()}, sys.path[:5]={sys.path[:5]}"
    ) from exc
registry = SourcesRegistry(source_path)
tier_counts = registry.tier_counts()
total_sources = sum(tier_counts.values())
blocked_domains = sorted(registry.blocked_domains)

print("Application SourcesRegistry path:", source_path)
print("Application SourcesRegistry tier_counts:", tier_counts)
print("Application SourcesRegistry total_sources:", total_sources)
print("Application SourcesRegistry blocked_domains:", blocked_domains[:20], "..." if len(blocked_domains) > 20 else "")

source_rows = []
for tier in range(1, 6):
    try:
        sources = registry.sources_for_tier(tier, hls_only=False)
    except Exception as exc:
        sources = []
        print(f"WARNING: sources_for_tier({tier}) failed: {exc}")
    source_rows.append({
        "max_tier": tier,
        "cumulative_sources": len(sources),
        "sample_sources": ", ".join(sources[:5]),
    })
display(pd.DataFrame(source_rows))

if total_sources == 0:
    msg = (
        "The application parsed ZERO sources from SOURCES.md. This matches the failed run symptom and means "
        "directory crawling will not have seed sources. Fix SOURCES.md structure/parser compatibility before using "
        "this run to evaluate camera discovery."
    )
    if FAIL_FAST_ON_ZERO_SOURCE_REGISTRY:
        raise RuntimeError(msg)
    else:
        print("WARNING:", msg)


,file,exists,bytes,mentions_public_only,mentions_hls_or_m3u8,mentions_no_synthetic_or_mock
0,AGENTS.md,True,17284,True,True,True
1,SKILLS.md,True,1763,True,True,False
2,SOURCES.md,True,5530,True,True,False
3,README.md,True,3669,True,True,True


Blocked/source-risk terms found in SOURCES.md: {'insecam': 2, 'shodan': 2, 'censys': 2, 'zoomeye': 2, 'fofa': 2}


2026-05-05 00:44:15.542 | INFO     | webcam_discovery.agents.directory_crawler:_parse:188 - SourcesRegistry: loaded tiers {1: 9, 2: 5, 3: 6} (20 total sources), 7 blocked domains from /content/webcam-discovery/SOURCES.md


Application SourcesRegistry path: /content/webcam-discovery/SOURCES.md
Application SourcesRegistry tier_counts: {1: 9, 2: 5, 3: 6}
Application SourcesRegistry total_sources: 20
Application SourcesRegistry blocked_domains: ['en.fofa.info', 'example.com', 'foto-webcam.eu', 'insecam.org', 'search.censys.io', 'shodan.io', 'zoomeye.org'] 


,max_tier,cumulative_sources,sample_sources
0,1,9,"https://www.windy.com/webcams, https://www.sky..."
1,2,14,"https://www.windy.com/webcams, https://www.sky..."
2,3,20,"https://www.windy.com/webcams, https://www.sky..."
3,4,20,"https://www.windy.com/webcams, https://www.sky..."
4,5,20,"https://www.windy.com/webcams, https://www.sky..."


## 04. Configure SSL and optional insecure fallback

TLS verification remains enabled by default. The fallback is off unless you explicitly set `ALLOW_INSECURE_SSL_FALLBACK = True` in the config cell.


In [4]:
import os, certifi, subprocess, sys

subprocess.run(["update-ca-certificates"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "certifi", "truststore"], check=False)

os.environ["WCD_CA_BUNDLE"] = certifi.where()
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
os.environ["CURL_CA_BUNDLE"] = certifi.where()

if ALLOW_INSECURE_SSL_FALLBACK:
    os.environ["WCD_ALLOW_INSECURE_SSL_FALLBACK"] = "true"
else:
    os.environ.pop("WCD_ALLOW_INSECURE_SSL_FALLBACK", None)

print("WCD_CA_BUNDLE =", os.environ["WCD_CA_BUNDLE"])
print("SSL_CERT_FILE =", os.environ["SSL_CERT_FILE"])
print("REQUESTS_CA_BUNDLE =", os.environ["REQUESTS_CA_BUNDLE"])
print("CURL_CA_BUNDLE =", os.environ["CURL_CA_BUNDLE"])
print("WCD_ALLOW_INSECURE_SSL_FALLBACK =", os.environ.get("WCD_ALLOW_INSECURE_SSL_FALLBACK", "<disabled>"))


WCD_CA_BUNDLE = /usr/local/lib/python3.12/dist-packages/certifi/cacert.pem
SSL_CERT_FILE = /usr/local/lib/python3.12/dist-packages/certifi/cacert.pem
REQUESTS_CA_BUNDLE = /usr/local/lib/python3.12/dist-packages/certifi/cacert.pem
CURL_CA_BUNDLE = /usr/local/lib/python3.12/dist-packages/certifi/cacert.pem
WCD_ALLOW_INSECURE_SSL_FALLBACK = <disabled>


## 05. Configure and preflight the real LLM planner

The app uses a real planner before discovery starts. This cell sets planner environment variables and sends a small preflight request. If this fails or times out, the notebook stops before the full run so the heartbeat does not sit at `starting` for a minute and then crash.


In [5]:
import os, time, json, requests

if USE_OLLAMA_CLOUD:
    try:
        from google.colab import userdata
        ollama_key = userdata.get(OLLAMA_SECRET_NAME) or ""
    except Exception:
        ollama_key = os.environ.get("WCD_OLLAMA_API_KEY", "") or os.environ.get("OLLAMA_API_KEY", "")

    os.environ["WCD_PLANNER_PROVIDER"] = PLANNER_PROVIDER
    os.environ["WCD_PLANNER_MODEL"] = PLANNER_MODEL
    os.environ["WCD_PLANNER_BASE_URL"] = PLANNER_BASE_URL
    os.environ["OLLAMA_HOST"] = PLANNER_BASE_URL
    os.environ["WCD_OLLAMA_API_KEY"] = ollama_key
    os.environ["OLLAMA_API_KEY"] = ollama_key
    # These timeout env vars are harmless if the current branch does not consume them; the preflight still validates connectivity.
    os.environ["WCD_PLANNER_TIMEOUT_SECONDS"] = str(OLLAMA_PREFLIGHT_TIMEOUT_SECONDS)
    os.environ["WCD_LLM_TIMEOUT_SECONDS"] = str(OLLAMA_PREFLIGHT_TIMEOUT_SECONDS)

    if not os.environ.get("WCD_OLLAMA_API_KEY"):
        raise RuntimeError(f"Missing {OLLAMA_SECRET_NAME}. Add it in Colab Secrets before running agentic discovery.")

print("Planner provider:", os.environ.get("WCD_PLANNER_PROVIDER"))
print("Planner model:", os.environ.get("WCD_PLANNER_MODEL"))
print("Planner base URL:", os.environ.get("WCD_PLANNER_BASE_URL"))
print("Planner key configured:", bool(os.environ.get("WCD_OLLAMA_API_KEY")))
print("Planner preflight timeout seconds:", OLLAMA_PREFLIGHT_TIMEOUT_SECONDS)

# Planner LLM preflight: small request, real API, no fake planner output.
def planner_llm_preflight():
    if not USE_OLLAMA_CLOUD:
        print("Skipping Ollama Cloud preflight because USE_OLLAMA_CLOUD=False")
        return {"skipped": True}
    url = PLANNER_BASE_URL.rstrip("/") + "/api/chat"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {os.environ['WCD_OLLAMA_API_KEY']}",
    }
    payload = {
        "model": PLANNER_MODEL,
        "stream": False,
        "messages": [
            {"role": "system", "content": "Return compact JSON only."},
            {"role": "user", "content": "Return {\"ok\": true, \"stage\": \"planner_preflight\"}."},
        ],
    }
    started = time.time()
    resp = requests.post(url, headers=headers, json=payload, timeout=OLLAMA_PREFLIGHT_TIMEOUT_SECONDS)
    elapsed = time.time() - started
    resp.raise_for_status()
    data = resp.json()
    content = ((data.get("message") or {}).get("content") or data.get("response") or "").strip()
    print(f"Planner LLM preflight succeeded in {elapsed:.1f}s")
    print("Preflight response preview:", content[:300])
    return {"elapsed_seconds": elapsed, "content_preview": content[:300]}

try:
    PLANNER_PREFLIGHT_RESULT = planner_llm_preflight()
except Exception as exc:
    msg = (
        "Planner LLM preflight failed before discovery started. This is the same failure class as the observed "
        "httpx.ReadTimeout during PlannerAgent.plan(). Fix model name, key, base URL, or timeout before running the full pipeline."
    )
    if FAIL_FAST_ON_LLM_PREFLIGHT_FAILURE:
        raise RuntimeError(msg) from exc
    else:
        print("WARNING:", msg, "\\n", repr(exc))
        PLANNER_PREFLIGHT_RESULT = {"error": repr(exc)}


Planner provider: ollama
Planner model: gemma4:31b-cloud
Planner base URL: https://ollama.com
Planner key configured: True
Planner preflight timeout seconds: 180
Planner LLM preflight succeeded in 1.1s
Preflight response preview: {"ok": true, "stage": "planner_preflight"}


## 06. Shared debug helpers

These helpers load JSON/JSONL artifacts, run subprocesses with an **active heartbeat monitor**, normalize fields across artifact formats, derive query-specific target scope, and perform generic URL metadata extraction. No helper is tied to a particular state, country, agency, or camera vendor.

The heartbeat monitor is intentionally verbose: it reports the current inferred stage, latest discovery lines, source/domain counts, validation counts, cap usage, location-leakage alerts, and a stop-file path. This is meant to help you stop a run early when the search drifts outside the user-requested target.


In [6]:
import os, sys, time, subprocess, threading, queue, pathlib, datetime, shlex, json, re, shutil, zipfile, math
from urllib.parse import urlparse, unquote
import pandas as pd
import requests



def _shorten(value, n=220):
    value = str(value or "").replace("\r", " ").replace("\n", " ").strip()
    return value[: n - 3] + "..." if len(value) > n else value


def _top_items(counter, n=10):
    return sorted(counter.items(), key=lambda kv: kv[1], reverse=True)[:n]


def _line_stage(line):
    text = (line or "").lower()
    if "sourcesregistry" in text:
        return "source-registry"
    if "planneragent" in text or "plan created" in text:
        return "planner"
    if "httpx.readtimeout" in text or "httpcore.readtimeout" in text:
        return "planner-or-http-timeout"
    if "searchagent" in text or "custom queries" in text or "search" in text and "query" in text:
        return "search"
    if "directoryagent" in text or "directorytraversalskill" in text or "crawling" in text:
        return "directory-crawl"
    if "extracting feeds" in text or "feedextraction" in text or "stream(s) found" in text:
        return "feed-extraction"
    if "validationagent" in text or "probing urls" in text:
        return "validation"
    if "ffprobe" in text:
        return "ffprobe"
    if "visual" in text and "analysis" in text:
        return "visual-analysis"
    if "llm geocoding" in text or "geocod" in text:
        return "geocoding"
    if "catalogagent" in text or "geojson" in text or "mapagent" in text or "map.html" in text:
        return "catalog-map"
    if "run-agentic complete" in text:
        return "complete"
    return None


def _extract_domain_from_line(line):
    urls = re.findall(r"https?://[^\s'\"<>]+", line or "")
    if urls:
        return urlparse(urls[0]).netloc.lower() or "unknown"
    m = re.search(r"from\s+([A-Za-z0-9.-]+)\s*\(", line or "")
    if m:
        return m.group(1).lower()
    m = re.search(r"crawling\s+([A-Za-z0-9.-]+)", line or "")
    if m:
        return m.group(1).lower()
    return "unknown"


def _update_heartbeat_state_from_line(state, line, limits=None):
    limits = limits or {}
    clean = _shorten(line, 500)
    lower = clean.lower()
    stage = _line_stage(clean)
    if stage:
        state["stage"] = stage
    state["total_lines"] = state.get("total_lines", 0) + 1

    m = re.search(r"SourcesRegistry:\s*loaded tiers\s*(\{.*?\})\s*\((\d+) total sources\)", clean, flags=re.I)
    if m:
        state["source_registry_total_sources"] = int(m.group(2))
        state["source_registry_tier_counts"] = m.group(1)
        if state["source_registry_total_sources"] == 0:
            state.setdefault("recent_useful_lines", []).append("WARNING: SourcesRegistry loaded ZERO sources; directory seed crawling cannot proceed normally.")
            state["recent_useful_lines"] = state["recent_useful_lines"][-50:]

    # Keep recent useful lines only; tqdm noise is intentionally compressed.
    useful_markers = [
        "planneragent", "searchagent", "directoryagent", "directorytraversalskill", "extracting feeds",
        "stream(s) found", "validationagent", "probing urls", "ffprobe", "llm geocoding",
        "catalogagent", "geojson", "mapagent", "run-agentic complete", "warning", "error",
        "candidate", "candidates", "records", "mapped=", "rejected=",
    ]
    if any(m in lower for m in useful_markers):
        state.setdefault("recent_useful_lines", []).append(clean)
        state["recent_useful_lines"] = state["recent_useful_lines"][-50:]

    # Search progress from tqdm-style lines.
    m = re.search(r"SearchAgent custom queries:\s*(\d+)%", clean)
    if m:
        state["search_progress_percent"] = int(m.group(1))
    m = re.search(r"SearchAgent custom queries:.*?(\d+)/(\d+)", clean)
    if m:
        state["search_queries_done"] = int(m.group(1))
        state["search_queries_total"] = int(m.group(2))

    # Directory crawl candidate lines.
    m = re.search(r"DirectoryTraversalSkill:\s*([0-9,]+)\s+candidates\s+from\s+([^\s]+)\s+\(([0-9,]+)\s+pages\)", clean, flags=re.I)
    if m:
        count = int(m.group(1).replace(",", ""))
        pages = int(m.group(3).replace(",", ""))
        domain = m.group(2).lower()
        state.setdefault("candidate_counts_by_domain", {})[domain] = state.setdefault("candidate_counts_by_domain", {}).get(domain, 0) + count
        state.setdefault("page_counts_by_domain", {})[domain] = state.setdefault("page_counts_by_domain", {}).get(domain, 0) + pages
        state["last_discovery"] = f"{count} candidate page(s) from {domain} across {pages} page(s)"

    # Candidate collapse lines.
    m = re.search(r"collapsed\s+([0-9,]+).*?\(([0-9,]+)\s*[→\-]+\s*([0-9,]+)\)", clean, flags=re.I)
    if m:
        state["collapsed_duplicates"] = int(m.group(1).replace(",", ""))
        state["candidates_before_collapse"] = int(m.group(2).replace(",", ""))
        state["candidates_after_collapse"] = int(m.group(3).replace(",", ""))

    # Stream count lines by source.
    m = re.search(r"([A-Za-z0-9.-]+):\s*([0-9,]+)\s+stream\(s\) found", clean, flags=re.I)
    if m:
        domain = m.group(1).lower()
        count = int(m.group(2).replace(",", ""))
        state.setdefault("streams_found_by_domain", {})[domain] = count
        state["last_discovery"] = f"{count} stream(s) found from {domain}"

    # Feed extraction progress from tqdm-style lines.
    m = re.search(r"Extracting feeds:\s*(\d+)%.*?([0-9,]+)/([0-9,]+)", clean)
    if m:
        state["feed_extract_percent"] = int(m.group(1))
        state["feed_extract_done"] = int(m.group(2).replace(",", ""))
        state["feed_extract_total"] = int(m.group(3).replace(",", ""))

    # Validation and stream-status lines.
    m = re.search(r"ValidationAgent:\s*([0-9,]+)\s+candidates received", clean, flags=re.I)
    if m:
        state["validation_candidates_received"] = int(m.group(1).replace(",", ""))
    m = re.search(r"probe results\s+[—-]\s+live=([0-9,]+),\s*dead=([0-9,]+),\s*unknown=([0-9,]+),\s*timeout=([0-9,]+),\s*other=([0-9,]+)", clean, flags=re.I)
    if m:
        state["probe_live"] = int(m.group(1).replace(",", ""))
        state["probe_dead"] = int(m.group(2).replace(",", ""))
        state["probe_unknown"] = int(m.group(3).replace(",", ""))
        state["probe_timeout"] = int(m.group(4).replace(",", ""))
        state["probe_other"] = int(m.group(5).replace(",", ""))
    m = re.search(r"ffprobe results\s+[—-]\s+live=([0-9,]+)\s+unknown=([0-9,]+)\s+dead=([0-9,]+)", clean, flags=re.I)
    if m:
        state["ffprobe_live"] = int(m.group(1).replace(",", ""))
        state["ffprobe_unknown"] = int(m.group(2).replace(",", ""))
        state["ffprobe_dead"] = int(m.group(3).replace(",", ""))
    m = re.search(r"([0-9,]+)\s+HLS streams to catalog\s+[—-]\s+live=([0-9,]+)\s+unknown=([0-9,]+)\s+dead=([0-9,]+)\s+dropped_dead=([0-9,]+)", clean, flags=re.I)
    if m:
        state["hls_to_catalog"] = int(m.group(1).replace(",", ""))
        state["catalog_live"] = int(m.group(2).replace(",", ""))
        state["catalog_unknown"] = int(m.group(3).replace(",", ""))
        state["catalog_dead"] = int(m.group(4).replace(",", ""))
        state["catalog_dropped_dead"] = int(m.group(5).replace(",", ""))
    m = re.search(r"([0-9,]+)\s+validated records\s+\(([0-9,]+)\s+with coords,\s+([0-9,]+)\s+without", clean, flags=re.I)
    if m:
        state["validated_records"] = int(m.group(1).replace(",", ""))
        state["validated_with_coords"] = int(m.group(2).replace(",", ""))
        state["validated_without_coords"] = int(m.group(3).replace(",", ""))
    m = re.search(r"CatalogAgent:\s*([0-9,]+)\s+unique records after dedup\s+\(dropped\s+([0-9,]+)\)", clean, flags=re.I)
    if m:
        state["catalog_unique_after_dedup"] = int(m.group(1).replace(",", ""))
        state["catalog_dropped_dedup"] = int(m.group(2).replace(",", ""))
    m = re.search(r"GeoJSONExportSkill:\s*([0-9,]+)\s+features written.*?([0-9,]+)\s+skipped", clean, flags=re.I)
    if m:
        state["geojson_features_written"] = int(m.group(1).replace(",", ""))
        state["geojson_features_skipped"] = int(m.group(2).replace(",", ""))
    m = re.search(r"run-agentic complete\s+[—-]\s+mapped=([0-9,]+),\s+needs_review_location_unknown=([0-9,]+),\s+rejected=([0-9,]+)", clean, flags=re.I)
    if m:
        state["mapped"] = int(m.group(1).replace(",", ""))
        state["needs_review_location_unknown"] = int(m.group(2).replace(",", ""))
        state["rejected"] = int(m.group(3).replace(",", ""))

    # URLs/domains seen in output.
    for url in re.findall(r"https?://[^\s'\"<>]+", clean):
        domain = urlparse(url).netloc.lower() or "unknown"
        state.setdefault("url_domains_seen", {})[domain] = state.setdefault("url_domains_seen", {}).get(domain, 0) + 1
        if any(marker in url.lower() for marker in ["m3u8", "playlist", "stream", "camera", "cam"]):
            state.setdefault("recent_cameraish_urls", []).append(url[:500])
            state["recent_cameraish_urls"] = state["recent_cameraish_urls"][-25:]

    return state


def _check_location_leakage(line, terms):
    hits = []
    text = (line or "").lower()
    for term in terms or []:
        t = str(term or "").strip().lower()
        if not t:
            continue
        # Multi-word phrases use substring; single tokens use word-ish boundaries.
        if " " in t:
            matched = t in text
        else:
            matched = re.search(r"(?<![a-z0-9])" + re.escape(t) + r"(?![a-z0-9])", text, flags=re.I) is not None
        if matched:
            hits.append(term)
    return hits


def _cap_usage_lines(state, limits):
    limits = limits or {}
    pairs = [
        ("feed_extract_total", "MAX_FEED_ENDPOINTS"),
        ("feed_extract_done", "MAX_FEED_ENDPOINTS"),
        ("validation_candidates_received", "MAX_STREAMS"),
        ("hls_to_catalog", "MAX_STREAMS"),
        ("mapped", "MAX_STREAMS"),
    ]
    lines = []
    for observed_key, limit_key in pairs:
        observed = state.get(observed_key)
        limit = limits.get(limit_key)
        if isinstance(observed, int) and isinstance(limit, int) and limit > 0:
            mark = "OVER" if observed > limit else "ok"
            lines.append(f"{observed_key}={observed}/{limit} {mark}")
    # Per-source stream cap applies to streams found by each source.
    per_source_limit = limits.get("PER_SOURCE_STREAM_CAP")
    if isinstance(per_source_limit, int) and per_source_limit > 0:
        over = [f"{d}={c}/{per_source_limit}" for d, c in _top_items(state.get("streams_found_by_domain", {}), 20) if c > per_source_limit]
        if over:
            lines.append("per_source_stream_cap OVER: " + ", ".join(over[:8]))
    return lines


def _print_heartbeat_summary(state, start, last_output, proc, stop_file=None, limits=None, recent_n=12, domain_top_n=12):
    now = time.time()
    elapsed_min = (now - start) / 60
    quiet_for = now - last_output
    stage = state.get("stage", "starting")
    print("\n" + "=" * 96, flush=True)
    print(f"[heartbeat] elapsed={elapsed_min:.1f} min stage={stage} pid={proc.pid} quiet_for={quiet_for:.0f}s lines={state.get('total_lines', 0)}", flush=True)
    if stop_file:
        print(f"[heartbeat] manual stop: create {stop_file} or interrupt this Colab cell", flush=True)
    planner_quiet_warning_seconds = globals().get("PLANNER_QUIET_WARNING_SECONDS", 30)
    if stage in {"starting", "source-registry", "planner"} and quiet_for >= planner_quiet_warning_seconds and not state.get("last_discovery"):
        print("[heartbeat] no discovery output yet: still before search/crawl. Check planner LLM connectivity and source registry.", flush=True)
    if state.get("source_registry_total_sources") == 0:
        print("[heartbeat] WARNING: SourcesRegistry reported 0 total sources. Directory crawling has no seed list.", flush=True)
    if state.get("last_discovery"):
        print(f"[heartbeat] latest discovery: {state['last_discovery']}", flush=True)

    # Progress / counts.
    progress_bits = []
    for key in [
        "search_progress_percent", "search_queries_done", "search_queries_total", "feed_extract_percent",
        "feed_extract_done", "feed_extract_total", "validation_candidates_received", "probe_live",
        "probe_dead", "probe_unknown", "ffprobe_live", "ffprobe_unknown", "ffprobe_dead",
        "hls_to_catalog", "validated_records", "validated_with_coords", "validated_without_coords",
        "catalog_unique_after_dedup", "geojson_features_written", "mapped", "needs_review_location_unknown", "rejected",
    ]:
        if key in state:
            progress_bits.append(f"{key}={state[key]}")
    if progress_bits:
        print("[heartbeat] counts: " + "; ".join(progress_bits), flush=True)

    cap_lines = _cap_usage_lines(state, limits or {})
    if cap_lines:
        print("[heartbeat] cap usage: " + " | ".join(cap_lines), flush=True)

    cand_top = _top_items(state.get("candidate_counts_by_domain", {}), domain_top_n)
    if cand_top:
        print("[heartbeat] candidate domains: " + ", ".join(f"{d}={c}" for d, c in cand_top), flush=True)
    stream_top = _top_items(state.get("streams_found_by_domain", {}), domain_top_n)
    if stream_top:
        print("[heartbeat] stream domains: " + ", ".join(f"{d}={c}" for d, c in stream_top), flush=True)
    url_top = _top_items(state.get("url_domains_seen", {}), domain_top_n)
    if url_top:
        print("[heartbeat] URL domains seen in logs: " + ", ".join(f"{d}={c}" for d, c in url_top), flush=True)

    if state.get("location_leakage_hits"):
        terms = ", ".join(sorted(set(state.get("location_leakage_terms_seen", []))))
        print(f"[LOCATION ALERT] hits={state.get('location_leakage_hits')} terms={terms}", flush=True)
        for row in state.get("recent_location_leakage_lines", [])[-min(5, recent_n):]:
            print(f"  - {row}", flush=True)

    urls = state.get("recent_cameraish_urls", [])[-min(5, recent_n):]
    if urls:
        print("[heartbeat] recent camera/stream URLs:", flush=True)
        for u in urls:
            print(f"  - {u}", flush=True)

    recent = state.get("recent_useful_lines", [])[-recent_n:]
    if recent:
        print("[heartbeat] recent useful output:", flush=True)
        for line in recent:
            print(f"  {line}", flush=True)
    print("=" * 96 + "\n", flush=True)


def _write_heartbeat_status(path, state, start, proc, limits=None):
    try:
        payload = {
            "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
            "elapsed_seconds": round(time.time() - start, 1),
            "pid": proc.pid,
            "returncode": proc.poll(),
            "stage": state.get("stage", "starting"),
            "state": state,
            "limits": limits or {},
        }
        pathlib.Path(path).write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    except Exception as exc:
        print(f"[heartbeat] could not write status JSON: {exc}", flush=True)


def run_with_heartbeat(
    cmd,
    cwd,
    log_path,
    env=None,
    heartbeat_seconds=30,
    target_positive_terms=None,
    target_negative_terms=None,
    extra_location_leakage_terms=None,
    monitor_location_leakage=True,
    auto_stop_on_location_leakage=False,
    location_leakage_stop_hits=3,
    location_leakage_stop_grace_seconds=45,
    max_runtime_minutes=0,
    stop_file=None,
    limits=None,
    recent_lines=12,
    domain_top_n=12,
    write_status_json=True,
):
    """Run a subprocess with active, stage-aware heartbeat output.

    This is a notebook monitor only. It does not change webcam-discovery logic. It can optionally
    terminate the child process if query-derived off-target terms appear repeatedly in live output.
    """
    log_path = pathlib.Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    status_path = log_path.parent / "heartbeat_status.json"
    q = queue.Queue()
    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env or os.environ.copy(),
    )

    def reader():
        assert proc.stdout is not None
        for line in proc.stdout:
            q.put(line)
        q.put(None)

    t = threading.Thread(target=reader, daemon=True)
    t.start()
    start = time.time()
    last_output = start
    last_heartbeat = 0
    state = {
        "stage": "starting",
        "total_lines": 0,
        "recent_useful_lines": [],
        "recent_cameraish_urls": [],
        "location_leakage_hits": 0,
        "location_leakage_terms_seen": [],
        "recent_location_leakage_lines": [],
        "target_positive_terms": list(target_positive_terms or [])[:50],
        "target_negative_terms": list(target_negative_terms or [])[:50],
    }
    leakage_terms = list(dict.fromkeys([str(t).strip() for t in (target_negative_terms or []) + (extra_location_leakage_terms or []) if str(t).strip()]))

    if stop_file:
        try:
            pathlib.Path(stop_file).unlink(missing_ok=True)
        except Exception:
            pass

    print("$ " + " ".join(shlex.quote(str(x)) for x in cmd))
    print(f"[heartbeat] started pid={proc.pid} at {datetime.datetime.now().isoformat(timespec='seconds')}")
    print(f"[heartbeat] interval={heartbeat_seconds}s; live status JSON={status_path if write_status_json else '<disabled>'}")
    if monitor_location_leakage:
        print(f"[heartbeat] monitoring off-target terms: {leakage_terms or '<none derived>'}")
        print(f"[heartbeat] auto-stop on leakage: {auto_stop_on_location_leakage} after {location_leakage_stop_hits} hit(s), grace={location_leakage_stop_grace_seconds}s")
    if stop_file:
        print(f"[heartbeat] manual stop file: {stop_file}")

    with log_path.open("w", encoding="utf-8") as log:
        while True:
            now = time.time()

            # Manual stop-file or runtime guard can terminate the child process.
            if stop_file and pathlib.Path(stop_file).exists() and proc.poll() is None:
                msg = f"[STOP] stop file detected at {stop_file}; terminating pid={proc.pid}"
                print(msg, flush=True); log.write(msg + "\n"); log.flush()
                proc.terminate()
                state["stop_reason"] = "manual_stop_file"
            if max_runtime_minutes and max_runtime_minutes > 0 and (now - start) / 60 >= max_runtime_minutes and proc.poll() is None:
                msg = f"[STOP] max runtime {max_runtime_minutes} min reached; terminating pid={proc.pid}"
                print(msg, flush=True); log.write(msg + "\n"); log.flush()
                proc.terminate()
                state["stop_reason"] = "max_runtime_minutes"

            try:
                item = q.get(timeout=1)
            except queue.Empty:
                now = time.time()
                if now - last_heartbeat >= heartbeat_seconds:
                    _print_heartbeat_summary(
                        state, start, last_output, proc, stop_file=stop_file, limits=limits,
                        recent_n=recent_lines, domain_top_n=domain_top_n,
                    )
                    if write_status_json:
                        _write_heartbeat_status(status_path, state, start, proc, limits=limits)
                    last_heartbeat = now
                if proc.poll() is not None and q.empty():
                    break
                continue

            if item is None:
                break

            last_output = time.time()
            print(item, end="", flush=True)
            log.write(item); log.flush()
            _update_heartbeat_state_from_line(state, item, limits=limits)

            if monitor_location_leakage and leakage_terms:
                hits = _check_location_leakage(item, leakage_terms)
                if hits:
                    state["location_leakage_hits"] += 1
                    state.setdefault("location_leakage_terms_seen", []).extend(hits)
                    state["location_leakage_terms_seen"] = list(dict.fromkeys(state["location_leakage_terms_seen"]))
                    alert = f"[LOCATION ALERT] off-target/ambiguous term(s) {hits} found in output: {_shorten(item, 350)}"
                    print(alert, flush=True)
                    log.write(alert + "\n"); log.flush()
                    state.setdefault("recent_location_leakage_lines", []).append(_shorten(item, 500))
                    state["recent_location_leakage_lines"] = state["recent_location_leakage_lines"][-25:]

                    elapsed_since_start = time.time() - start
                    if (
                        auto_stop_on_location_leakage
                        and elapsed_since_start >= location_leakage_stop_grace_seconds
                        and state["location_leakage_hits"] >= location_leakage_stop_hits
                        and proc.poll() is None
                    ):
                        stop_msg = (
                            f"[STOP] auto-stop triggered by location leakage: "
                            f"hits={state['location_leakage_hits']} terms={state['location_leakage_terms_seen']}"
                        )
                        print(stop_msg, flush=True)
                        log.write(stop_msg + "\n"); log.flush()
                        state["stop_reason"] = "location_leakage"
                        proc.terminate()

            now = time.time()
            if now - last_heartbeat >= heartbeat_seconds:
                _print_heartbeat_summary(
                    state, start, last_output, proc, stop_file=stop_file, limits=limits,
                    recent_n=recent_lines, domain_top_n=domain_top_n,
                )
                if write_status_json:
                    _write_heartbeat_status(status_path, state, start, proc, limits=limits)
                last_heartbeat = now

        try:
            rc = proc.wait(timeout=20)
        except subprocess.TimeoutExpired:
            print(f"[STOP] process did not exit after terminate; killing pid={proc.pid}", flush=True)
            proc.kill()
            rc = proc.wait()
        elapsed = time.time() - start
        state["returncode"] = rc
        state["elapsed_seconds"] = round(elapsed, 1)
        if write_status_json:
            _write_heartbeat_status(status_path, state, start, proc, limits=limits)
        msg = f"[exit={rc}, elapsed={elapsed:.1f}s, status_json={status_path}]"
        print(msg)
        log.write(msg + "\n")
        return rc, elapsed

def load_json(path):
    path = pathlib.Path(path)
    if not path.exists():
        return None
    try:
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as exc:
        return {"_error": str(exc), "_path": str(path)}


def load_jsonl(path):
    path = pathlib.Path(path)
    rows = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8", errors="replace") as f:
        for lineno, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
                if isinstance(row, dict):
                    row["_artifact"] = str(path.name)
                    row["_line"] = lineno
                    rows.append(row)
                else:
                    rows.append({"_raw": row, "_artifact": str(path.name), "_line": lineno})
            except Exception:
                rows.append({"_raw": line, "_artifact": str(path.name), "_line": lineno})
    return rows


def flatten(prefix, obj, out):
    if isinstance(obj, dict):
        for k, v in obj.items():
            flatten(f"{prefix}.{k}" if prefix else str(k), v, out)
    elif isinstance(obj, list):
        out[prefix + ".len"] = len(obj)
    else:
        out[prefix] = obj
    return out


def first_value(row, keys, default=""):
    for k in keys:
        v = row.get(k)
        if v not in (None, ""):
            return v
    return default


def row_url(row):
    return str(first_value(row, ["candidate_url", "stream_url", "url", "feed_url", "page_url", "source_url", "manifest_url"], ""))


def row_text(row):
    try:
        return json.dumps(row, ensure_ascii=False, default=str).lower()
    except Exception:
        return str(row).lower()


def row_domain(row):
    url = row_url(row)
    return urlparse(url).netloc.lower() if url else "unknown"


def artifact_jsonl_files(base_dir):
    base_dir = pathlib.Path(base_dir)
    paths = []
    for sub in ["logs", "candidates", "snapshots"]:
        d = base_dir / sub
        if d.exists():
            paths.extend(sorted(d.rglob("*.jsonl")))
    return paths


def load_all_jsonl(base_dir):
    rows = []
    for p in artifact_jsonl_files(base_dir):
        for r in load_jsonl(p):
            r["_artifact_rel"] = str(p.relative_to(base_dir))
            r["domain"] = row_domain(r)
            rows.append(r)
    return rows


def extract_json_object(text):
    """Best-effort extraction of a JSON object from an LLM response."""
    if not text:
        return None
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text, flags=re.I).strip()
        text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end + 1])
        except Exception:
            return None
    return None


def _env_int(name, default):
    try:
        return int(os.environ.get(name, default))
    except Exception:
        return int(default)


def ollama_chat_json(prompt, model=None, timeout=None, *, label="llm-json"):
    """Call Ollama Cloud/local Ollama-compatible chat API and return parsed JSON.

    This function intentionally does NOT use heuristic fallback. It only makes the
    LLM JSON call more reliable by using structured-output mode, deterministic
    generation options, connect/read timeouts, and bounded retries for transient
    network/cloud read timeouts.
    """
    import requests
    from requests.exceptions import ConnectionError, HTTPError, ReadTimeout, Timeout

    model = model or os.environ.get("LLM_EXTRACTION_MODEL") or os.environ.get("WCD_PLANNER_MODEL") or PLANNER_MODEL
    api_key = os.environ.get("WCD_OLLAMA_API_KEY") or os.environ.get("OLLAMA_API_KEY") or ""
    host = os.environ.get("OLLAMA_HOST") or ("https://ollama.com" if api_key else "http://localhost:11434")
    url = host.rstrip("/") + "/api/chat"
    headers = {"Content-Type": "application/json"}
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"

    connect_timeout = _env_int("OLLAMA_EXTRACTION_CONNECT_TIMEOUT_SECONDS", globals().get("OLLAMA_EXTRACTION_CONNECT_TIMEOUT_SECONDS", 20))
    read_timeout = _env_int("OLLAMA_EXTRACTION_READ_TIMEOUT_SECONDS", timeout or globals().get("OLLAMA_EXTRACTION_READ_TIMEOUT_SECONDS", 240))
    max_attempts = max(1, _env_int("OLLAMA_EXTRACTION_MAX_ATTEMPTS", globals().get("OLLAMA_EXTRACTION_MAX_ATTEMPTS", 3)))
    backoff_seconds = max(0, _env_int("OLLAMA_EXTRACTION_BACKOFF_SECONDS", globals().get("OLLAMA_EXTRACTION_BACKOFF_SECONDS", 8)))

    payload = {
        "model": model,
        "stream": False,
        "format": "json",
        "think": False,
        "options": {
            "temperature": 0,
            "top_p": 0.1,
            "num_predict": 700,
        },
        "messages": [
            {"role": "system", "content": "Return only valid compact JSON. Do not include prose or markdown."},
            {"role": "user", "content": prompt},
        ],
    }

    last_exc = None
    retryable_statuses = {408, 409, 425, 429, 500, 502, 503, 504}
    for attempt in range(1, max_attempts + 1):
        started = time.time()
        try:
            print(f"[{label}] LLM JSON call attempt {attempt}/{max_attempts}: model={model}, connect_timeout={connect_timeout}s, read_timeout={read_timeout}s")
            resp = requests.post(url, headers=headers, json=payload, timeout=(connect_timeout, read_timeout))
            if resp.status_code in retryable_statuses and attempt < max_attempts:
                raise HTTPError(f"retryable HTTP {resp.status_code}: {resp.text[:300]}", response=resp)
            resp.raise_for_status()
            data = resp.json()
            content = ((data.get("message") or {}).get("content") or "").strip()
            parsed = extract_json_object(content)
            if parsed is None:
                raise ValueError(f"LLM response was not valid JSON: {content[:500]}")
            elapsed = time.time() - started
            print(f"[{label}] LLM JSON call succeeded in {elapsed:.1f}s")
            return parsed
        except (ReadTimeout, Timeout, ConnectionError, HTTPError) as exc:
            last_exc = exc
            elapsed = time.time() - started
            print(f"[{label}] LLM JSON call attempt {attempt}/{max_attempts} failed after {elapsed:.1f}s: {type(exc).__name__}: {_shorten(exc, 300)}")
            if isinstance(exc, HTTPError):
                response = getattr(exc, "response", None)
                status_code = getattr(response, "status_code", None)
                if status_code not in retryable_statuses:
                    break
            if attempt < max_attempts:
                sleep_for = backoff_seconds * attempt
                print(f"[{label}] retrying after {sleep_for}s; extraction remains LLM-only, no heuristic fallback will be used")
                time.sleep(sleep_for)
        except Exception as exc:
            # Invalid JSON/schema/content errors are not silently repaired with heuristics.
            last_exc = exc
            print(f"[{label}] LLM JSON call failed with non-retryable error: {type(exc).__name__}: {_shorten(exc, 300)}")
            break

    raise RuntimeError(
        f"{label} failed after {max_attempts} LLM attempt(s). "
        "Heuristic fallback is intentionally disabled. "
        "Check Ollama Cloud availability/key/model or switch TARGET_SCOPE_MODE='MANUAL' for a query-specific manual scope."
    ) from last_exc


def derive_target_scope(query):
    """Derive target scope using the configured LLM only, unless MANUAL mode is selected.

    No heuristic fallback is permitted. If LLM extraction fails, the notebook stops so
    the failure is visible before a broad/off-target discovery run starts.
    """
    if TARGET_SCOPE_MODE == "MANUAL":
        scope = dict(MANUAL_TARGET_SCOPE)
        scope["mode"] = "manual"
        scope.setdefault("confidence", 1.0)
        return scope

    if TARGET_SCOPE_MODE != "LLM_FROM_QUERY":
        raise RuntimeError(
            f"Unsupported TARGET_SCOPE_MODE={TARGET_SCOPE_MODE!r}. "
            "Use LLM_FROM_QUERY or MANUAL. Heuristic target-scope parsing is disabled."
        )

    if not ENABLE_LLM_TARGET_SCOPE_AUDIT:
        raise RuntimeError(
            "ENABLE_LLM_TARGET_SCOPE_AUDIT is False, but heuristic target-scope parsing is disabled. "
            "Enable LLM target-scope extraction or use TARGET_SCOPE_MODE='MANUAL'."
        )

    prompt = f"""
Parse this webcam-discovery user query into a target-scope JSON object.

Query: {query!r}

Rules:
- The user query is the only source of truth.
- Do not default to California, the United States, or any other place unless the query says so.
- If the query does not contain a geocodable place, landmark, location, IP/location clue, or other specific location indicator, set requested_places=[] and confidence below 0.50.
- If a place name is ambiguous, include likely wrong same-name locations in negative_terms or ambiguous_same_name_terms.
- Use bbox only if you are confident for the requested place. Otherwise set bbox to null.
- Return only JSON with keys:
  requested_places: list[str]
  country_terms: list[str]
  admin1_terms: list[str]
  admin2_terms: list[str]
  locality_terms: list[str]
  landmark_terms: list[str]
  positive_terms: list[str]
  negative_terms: list[str]
  ambiguous_same_name_terms: list[str]
  bbox: null or {{label: str, south: number, west: number, north: number, east: number}}
  confidence: number between 0 and 1
  notes: str
"""
    try:
        scope = ollama_chat_json(prompt, label="target-scope-extraction")
    except Exception as exc:
        raise RuntimeError(
            "LLM target-scope extraction failed. Heuristic fallback is intentionally disabled."
        ) from exc

    if not isinstance(scope, dict):
        raise RuntimeError(f"LLM target-scope extraction returned non-object JSON: {type(scope).__name__}")

    scope["mode"] = "llm_from_query"

    confidence = scope.get("confidence", 0)
    try:
        confidence_value = float(confidence)
    except Exception:
        confidence_value = 0.0

    requested_places = scope.get("requested_places") or []
    if isinstance(requested_places, str):
        requested_places = [requested_places]

    if confidence_value < 0.50 or not [p for p in requested_places if str(p).strip()]:
        raise RuntimeError(
            "The query does not contain enough confident location information to run discovery. "
            f"LLM scope={json.dumps(scope, ensure_ascii=False)[:2000]}"
        )

    return scope


def scope_terms(scope, key):
    values = scope.get(key) or []
    if isinstance(values, str):
        values = [values]
    return [str(v).strip().lower() for v in values if str(v).strip()]


def positive_scope_terms(scope):
    terms = []
    for key in ["requested_places", "country_terms", "admin1_terms", "admin2_terms", "locality_terms", "landmark_terms", "positive_terms"]:
        terms.extend(scope_terms(scope, key))
    return list(dict.fromkeys(terms))


def negative_scope_terms(scope):
    terms = []
    for key in ["negative_terms", "ambiguous_same_name_terms"]:
        terms.extend(scope_terms(scope, key))
    return list(dict.fromkeys(terms))


def in_bbox(lon, lat, bbox):
    if not bbox:
        return None
    try:
        lon = float(lon); lat = float(lat)
    except Exception:
        return False
    return bbox["west"] <= lon <= bbox["east"] and bbox["south"] <= lat <= bbox["north"]



# URL metadata extraction is intentionally LLM-only.
# URL collection may still use basic string/regex discovery to gather real artifact URLs,
# but district/route/direction/place metadata must be extracted by llm_extract_url_metadata().


def llm_extract_url_metadata(url, source_context, target_scope):
    """Extract camera URL metadata using the configured LLM only.

    This replaces the prior generic heuristic URL parser. The LLM must return
    evidence/confidence and mark ambiguity instead of silently accepting weak clues.
    """
    prompt = f"""
Extract camera metadata from this public webcam/HLS URL and optional artifact context.

URL: {url}
Target scope from user query: {json.dumps(target_scope, ensure_ascii=False)[:4000]}
Artifact context: {str(source_context)[:3000]}

Rules:
- Do not use source-specific hardcoded parsing.
- Do not assume California, Caltrans, WZMedia, the United States, or any default location.
- Infer only metadata supported by the URL/context.
- If a token may mean multiple places, mark ambiguous=true and explain.
- Preserve unknown fields as empty strings rather than inventing values.
- Return only JSON with keys:
  district_or_region: string
  route: string
  direction: string
  road_or_street: string
  locality: string
  admin_area: string
  country: string
  agency_or_source: string
  evidence: list[str]
  ambiguous: boolean
  confidence: number between 0 and 1
  notes: string
"""
    parsed = ollama_chat_json(prompt, label="url-metadata-extraction")
    if not isinstance(parsed, dict):
        raise RuntimeError(f"LLM URL metadata extraction returned non-object JSON for {url}: {type(parsed).__name__}")
    parsed["parser"] = "llm_url_metadata_v1"
    return parsed

warnings_accumulator = []

def warn(msg):
    warnings_accumulator.append(msg)
    print("WARNING:", msg)


## 07. Step-by-step CLI capability check

This cell records whether expected flags exist. It also checks for recommended future flags that would help enforce query-derived target scope, candidate caps, and metadata extraction without tying the app to any specific location or camera vendor.


In [7]:
help_proc = subprocess.run(["webcam-discovery", "run-agentic", "--help"], cwd=repo_dir, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
help_text = help_proc.stdout
expected_flags = [
    "--output-dir", "--enable-feed-discovery", "--max-feed-endpoints", "--max-feed-records",
    "--max-stream-candidates", "--per-source-stream-cap", "--preserve-direct-streams",
    "--max-streams", "--max-search-queries", "--max-search-results-per-query",
    "--enable-visual-analysis", "--enable-memory",
    "--disable-ffprobe-validation", "--disable-validation",
]
planner_cli_flags = ["--llm-provider", "--llm-model", "--planner-provider", "--planner-model"]
recommended_future_flags = [
    "--strict-target-scope", "--target-scope-json", "--target-place", "--target-country",
    "--target-admin1", "--target-admin2", "--location-confidence-threshold",
    "--max-total-candidates", "--max-validation-candidates", "--metadata-extraction-provider",
]
flag_rows = []
for f in expected_flags + planner_cli_flags + recommended_future_flags:
    flag_rows.append({
        "flag": f,
        "present": f in help_text,
        "category": "expected_current" if f in expected_flags else ("planner_cli_optional" if f in planner_cli_flags else "recommended_future"),
    })
display(pd.DataFrame(flag_rows))

missing_expected = [r["flag"] for r in flag_rows if r["category"] == "expected_current" and not r["present"]]
if missing_expected:
    message = "Missing expected CLI flags: " + ", ".join(missing_expected)
    if "--disable-ffprobe-validation" in missing_expected or "--disable-validation" in missing_expected:
        raise RuntimeError(message)
    warn(message)

scope_artifact_names = [
    "logs/scope_inference.json",
    "logs/scope_summary.json",
    "logs/search_result_scope_decisions.jsonl",
    "logs/stream_candidate_scope_decisions.jsonl",
]
print("Expected LLM scope artifacts:", scope_artifact_names)


,flag,present,category
0,--output-dir,True,expected_current
1,--enable-feed-discovery,True,expected_current
2,--max-feed-endpoints,True,expected_current
3,--max-feed-records,True,expected_current
4,--max-stream-candidates,True,expected_current
5,--per-source-stream-cap,True,expected_current
6,--preserve-direct-streams,True,expected_current
7,--max-streams,True,expected_current
8,--max-search-queries,True,expected_current
9,--max-search-results-per-query,True,expected_current


## 08. Query-derived target-scope sanity check before running

This notebook does not use a California/default bounding box. It derives the target scope from the user query, optionally using an LLM, and uses that derived scope to audit planner/search/candidate/catalog artifacts for leakage.


In [8]:
# This pre-run LLM extraction is only a guardrail/display aid.
# The authoritative scope-enforcement validation below reads the real app artifacts.
TARGET_SCOPE = derive_target_scope(QUERY)
TARGET_POSITIVE_TERMS = positive_scope_terms(TARGET_SCOPE)
TARGET_NEGATIVE_TERMS = negative_scope_terms(TARGET_SCOPE)
# Region terms are positive geography terms used by later search-result audits.
TARGET_REGION_TERMS = list(dict.fromkeys(
    scope_terms(TARGET_SCOPE, "country_terms")
    + scope_terms(TARGET_SCOPE, "admin1_terms")
    + scope_terms(TARGET_SCOPE, "admin2_terms")
    + scope_terms(TARGET_SCOPE, "locality_terms")
))
TARGET_LEAKAGE_TERMS = list(dict.fromkeys(TARGET_NEGATIVE_TERMS + [str(t).strip().lower() for t in EXTRA_LOCATION_LEAKAGE_TERMS if str(t).strip()]))
TARGET_BBOX = TARGET_SCOPE.get("bbox") if isinstance(TARGET_SCOPE, dict) else None

print("Derived TARGET_SCOPE:")
print(json.dumps(TARGET_SCOPE, indent=2, ensure_ascii=False))
print("Heartbeat leakage-monitor terms:", TARGET_LEAKAGE_TERMS or "<none>")

scope_rows = []
q_lower = QUERY.lower()
for term in TARGET_POSITIVE_TERMS:
    scope_rows.append({"term": term, "found_in_query": term.lower() in q_lower, "type": "positive/target"})
for term in TARGET_NEGATIVE_TERMS:
    scope_rows.append({"term": term, "found_in_query": term.lower() in q_lower, "type": "negative/leakage_or_ambiguous"})

scope_df = pd.DataFrame(scope_rows)
display(scope_df if not scope_df.empty else pd.DataFrame([{"note": "No target terms derived. Use LLM_FROM_QUERY with a valid key or MANUAL mode."}]))

if not TARGET_POSITIVE_TERMS:
    raise RuntimeError("No positive target terms were derived by the LLM. Discovery should not run without a confident query-derived target scope.")

if TARGET_BBOX:
    print("Query-derived bbox audit enabled:", TARGET_BBOX)
else:
    print("No bbox audit enabled. This is expected unless the query-derived/manual target scope includes a confident bbox.")

if MONITOR_LOCATION_LEAKAGE and AUTO_STOP_ON_LOCATION_LEAKAGE and not TARGET_LEAKAGE_TERMS:
    warn("Auto-stop on location leakage is enabled, but no leakage terms were derived. Add query-specific EXTRA_LOCATION_LEAKAGE_TERMS or enable/fix LLM target-scope audit before relying on auto-stop.")


[target-scope-extraction] LLM JSON call attempt 1/3: model=gemma4:31b-cloud, connect_timeout=20s, read_timeout=240s
[target-scope-extraction] LLM JSON call succeeded in 2.4s
Derived TARGET_SCOPE:
{
  "requested_places": [
    "Cameron, Texas"
  ],
  "country_terms": [
    "USA"
  ],
  "admin1_terms": [
    "Texas"
  ],
  "admin2_terms": [],
  "locality_terms": [
    "Cameron"
  ],
  "landmark_terms": [],
  "positive_terms": [
    "traffic cameras"
  ],
  "negative_terms": [],
  "ambiguous_same_name_terms": [],
  "bbox": null,
  "confidence": 1.0,
  "notes": "User specifically requested traffic cameras in Cameron, Texas.",
  "mode": "llm_from_query"
}
Heartbeat leakage-monitor terms: <none>


,term,found_in_query,type
0,"cameron, texas",True,positive/target
1,usa,False,positive/target
2,texas,True,positive/target
3,cameron,True,positive/target
4,traffic cameras,True,positive/target


No bbox audit enabled. This is expected unless the query-derived/manual target scope includes a confident bbox.


## 09. Run the real application once with explicit limits

This is the only full application run. The rest of the notebook breaks down the generated artifacts by stage.

The heartbeat is now stage-aware and discovery-aware. While the subprocess is still running, it prints:

- inferred stage: planner, search, directory crawl, feed extraction, validation, ffprobe, geocoding, catalog/map
- latest discovery lines and recent camera/stream URLs
- candidate counts by source/domain and stream counts by source/domain
- validation and catalog counts as soon as they appear
- cap usage against the configured limits
- query-derived location-leakage alerts
- a manual stop-file path and an optional auto-stop-on-leakage guard

This makes it easier to interrupt a run early if a query such as `Cameron, Texas` starts accepting off-target results from similarly named places.


In [9]:
import pathlib, re, shutil, os, json, datetime

slug = re.sub(r"[^a-zA-Z0-9]+", "-", QUERY.lower()).strip("-")[:100] or "run"
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir = pathlib.Path(ARTIFACT_ROOT) / f"{RUN_LABEL}_{LIMIT_PRESET.lower()}_{stamp}_{slug}"
if run_dir.exists():
    shutil.rmtree(run_dir)
run_dir.mkdir(parents=True, exist_ok=True)

# Clear repo-root runtime artifacts so this run is clean.
for p in ["logs", "candidates", "snapshots", "camera.geojson", "cameras.md", "map.html"]:
    target = repo_dir / p
    if target.is_dir():
        shutil.rmtree(target)
    elif target.exists():
        target.unlink()

cmd = [
    "webcam-discovery", "run-agentic", QUERY,
    "--output-dir", str(run_dir),
]

if IGNORE_SOURCES_MD:
    cmd.append("--ignore-sources-md")
if ENABLE_FEED_DISCOVERY:
    cmd.append("--enable-feed-discovery")
cmd += [
    "--max-feed-endpoints", str(MAX_FEED_ENDPOINTS),
    "--max-feed-records", str(MAX_FEED_RECORDS),
    "--max-stream-candidates", str(MAX_STREAM_CANDIDATES),
    "--per-source-stream-cap", str(PER_SOURCE_STREAM_CAP),
]
if PRESERVE_DIRECT_STREAMS:
    cmd.append("--preserve-direct-streams")
cmd += [
    "--max-streams", str(MAX_STREAMS),
    "--max-search-queries", str(MAX_SEARCH_QUERIES),
    "--max-search-results-per-query", str(MAX_SEARCH_RESULTS_PER_QUERY),
]

# Pass all SSL/planner env vars to subprocess before CLI fallback flags may modify env.
env = os.environ.copy()
env.setdefault("WCD_PLANNER_PROVIDER", PLANNER_PROVIDER)
env.setdefault("WCD_PLANNER_MODEL", PLANNER_MODEL)
env.setdefault("WCD_PLANNER_BASE_URL", PLANNER_BASE_URL)
env.setdefault("OLLAMA_HOST", PLANNER_BASE_URL)
env.setdefault("WCD_PLANNER_TIMEOUT_SECONDS", str(OLLAMA_PREFLIGHT_TIMEOUT_SECONDS))
env.setdefault("WCD_LLM_TIMEOUT_SECONDS", str(OLLAMA_PREFLIGHT_TIMEOUT_SECONDS))

# Use explicit planner CLI flags only when the branch supports them; otherwise env vars below control the planner.
if "--llm-provider" in help_text:
    cmd += ["--llm-provider", PLANNER_PROVIDER]
elif "--planner-provider" in help_text:
    cmd += ["--planner-provider", PLANNER_PROVIDER]
if "--llm-model" in help_text:
    cmd += ["--llm-model", PLANNER_MODEL]
elif "--planner-model" in help_text:
    cmd += ["--planner-model", PLANNER_MODEL]

if ENABLE_MEMORY:
    cmd.append("--enable-memory")
if ENABLE_VISUAL_ANALYSIS and not DISABLE_VALIDATION:
    cmd.append("--enable-visual-analysis")

if DISABLE_FFPROBE_VALIDATION:
    if "--disable-ffprobe-validation" in help_text:
        cmd.append("--disable-ffprobe-validation")
    else:
        env["WCD_USE_FFPROBE_VALIDATION"] = "false"

if DISABLE_VALIDATION:
    if "--disable-validation" not in help_text:
        raise RuntimeError("DISABLE_VALIDATION=True, but --disable-validation is not available in the CLI.")
    cmd.append("--disable-validation")
run_log = run_dir / "notebook_run_stdout_stderr.log"
heartbeat_limits = {
    "MAX_FEED_ENDPOINTS": MAX_FEED_ENDPOINTS,
    "MAX_FEED_RECORDS": MAX_FEED_RECORDS,
    "MAX_STREAM_CANDIDATES": MAX_STREAM_CANDIDATES,
    "PER_SOURCE_STREAM_CAP": PER_SOURCE_STREAM_CAP,
    "MAX_STREAMS": MAX_STREAMS,
    "MAX_SEARCH_QUERIES": MAX_SEARCH_QUERIES,
    "MAX_SEARCH_RESULTS_PER_QUERY": MAX_SEARCH_RESULTS_PER_QUERY,
}

rc, elapsed = run_with_heartbeat(
    cmd,
    cwd=repo_dir,
    log_path=run_log,
    env=env,
    heartbeat_seconds=HEARTBEAT_SECONDS,
    target_positive_terms=TARGET_POSITIVE_TERMS,
    target_negative_terms=TARGET_NEGATIVE_TERMS,
    extra_location_leakage_terms=EXTRA_LOCATION_LEAKAGE_TERMS,
    monitor_location_leakage=MONITOR_LOCATION_LEAKAGE,
    auto_stop_on_location_leakage=AUTO_STOP_ON_LOCATION_LEAKAGE,
    location_leakage_stop_hits=LOCATION_LEAKAGE_STOP_HITS,
    location_leakage_stop_grace_seconds=LOCATION_LEAKAGE_STOP_GRACE_SECONDS,
    max_runtime_minutes=MAX_RUNTIME_MINUTES,
    stop_file=STOP_FILE,
    limits=heartbeat_limits,
    recent_lines=HEARTBEAT_RECENT_LINES,
    domain_top_n=HEARTBEAT_SHOW_DOMAIN_TOP_N,
    write_status_json=HEARTBEAT_STATUS_JSON,
)
print("RUN_DIR:", run_dir)
print("RETURN_CODE:", rc)
print("ELAPSED_SECONDS:", elapsed)

if rc != 0:
    warn(f"Application exited with non-zero status: {rc}. Continue artifact audit if partial artifacts exist.")


Streaming output truncated to the last 5000 lines.
  - https://www.traffic-cams.com/texas/all:
  - https://www.traffic-cams.com/texas/all:
[heartbeat] recent useful output:
  ffprobe:  57%|█████▋    | 1258/2192 [39:30<37:25,  2.40s/url]

ffprobe:  58%|█████▊    | 1262/2192 [39:43<41:31,  2.68s/url]

[heartbeat] elapsed=46.8 min stage=ffprobe pid=27572 quiet_for=1s lines=1650
[heartbeat] manual stop: create /content/STOP_WEBCAM_DISCOVERY or interrupt this Colab cell
[heartbeat] counts: search_progress_percent=50; search_queries_done=9; search_queries_total=18; validation_candidates_received=2192
[heartbeat] cap usage: validation_candidates_received=2192/2500 ok
[heartbeat] URL domains seen in logs: www.traffic-cams.com=3
[heartbeat] recent camera/stream URLs:
  - https://www.traffic-cams.com/texas/all:
  - https://www.traffic-cams.com/texas/all:
  - https://www.traffic-cams.com/texas/all:
[heartbeat] recent useful output:
  ffprobe:  58%|█████▊    | 1262/2192 [39:43<41:31,  2.68s/url]



ValueError: Invalid IPv6 URL

## 10. Copy repo-root artifacts into the run directory

Some versions of the app write artifacts under `--output-dir`; others also write to the repository root. This cell copies both so review is consistent.


In [ ]:
for name in ["logs", "candidates", "snapshots"]:
    src = repo_dir / name
    dst = run_dir / name
    if src.exists():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
for name in ["camera.geojson", "cameras.md", "map.html"]:
    src = repo_dir / name
    if src.exists():
        shutil.copy2(src, run_dir / name)

inventory = []
for p in sorted(run_dir.rglob("*")):
    if p.is_file():
        inventory.append({"artifact": str(p.relative_to(run_dir)), "bytes": p.stat().st_size})
inv_df = pd.DataFrame(inventory)
display(inv_df)
print("Run directory:", run_dir)


## LLM Scope-Enforcement Validation

Validate the real application's LLM scope inference and LLM scope-gate artifacts before auditing expensive validation/catalog stages.


In [ ]:
scope_inference = load_json(run_dir / "logs" / "scope_inference.json") or {}
scope_summary = load_json(run_dir / "logs" / "scope_summary.json") or {}
search_scope_rows = load_jsonl(run_dir / "logs" / "search_result_scope_decisions.jsonl")
stream_scope_rows = load_jsonl(run_dir / "logs" / "stream_candidate_scope_decisions.jsonl")
run_summary = load_json(run_dir / "logs" / "run_summary.json") or {}

search_scope_counts = {}
stream_scope_counts = {}

if ENABLE_LLM_SCOPE_ENFORCEMENT_VALIDATION:
    missing = []
    for rel in [
        "logs/scope_inference.json",
        "logs/scope_summary.json",
        "logs/search_result_scope_decisions.jsonl",
        "logs/stream_candidate_scope_decisions.jsonl",
    ]:
        if not (run_dir / rel).exists():
            missing.append(rel)
    if missing:
        raise RuntimeError("Missing LLM scope-enforcement artifacts: " + ", ".join(missing))

    if not scope_inference:
        raise RuntimeError("scope_inference.json is empty; scope enforcement was not validated.")

    if scope_inference.get("has_sufficient_scope") is False:
        print("Application identified insufficient scope and should have stopped before discovery.")
    else:
        print("Scope inferred by application:")
        print(json.dumps(scope_inference, indent=2, ensure_ascii=False)[:4000])

    for name, rows in [
        ("search_result_scope_decisions", search_scope_rows),
        ("stream_candidate_scope_decisions", stream_scope_rows),
    ]:
        if rows:
            df = pd.DataFrame(rows)
            if "decision" in df.columns:
                counts_df = df.groupby("decision").size().reset_index(name="count")
                display(counts_df)
                counts = dict(zip(counts_df["decision"], counts_df["count"]))
                if name.startswith("search"):
                    search_scope_counts = counts
                else:
                    stream_scope_counts = counts
            else:
                display(df.head())
            bad_rows = [r for r in rows if not r.get("reason") or r.get("confidence") is None]
            if bad_rows:
                warn(f"{name} has rows without reason/confidence; scope audit is incomplete.")
            no_model_rows = [r for r in rows if not (r.get("provider") or r.get("model") or r.get("raw_llm_response"))]
            if no_model_rows:
                warn(f"{name} has rows without provider/model/raw LLM metadata.")
        else:
            warn(f"{name} is empty. This is expected only if discovery stopped early or no candidates were produced.")

    summary_scope = run_summary.get("scope") or scope_summary.get("scope") or {}
    expected_count_keys = [
        "search_results_accepted", "search_results_rejected", "search_results_review",
        "stream_candidates_accepted", "stream_candidates_rejected", "stream_candidates_review",
    ]
    missing_count_keys = [k for k in expected_count_keys if k not in summary_scope]
    if missing_count_keys:
        raise RuntimeError("Run summary/scope summary missing scope gate counts: " + ", ".join(missing_count_keys))

    if scope_inference.get("has_sufficient_scope") is False:
        broad_artifacts = [
            "logs/search_queries.jsonl",
            "logs/search_results.jsonl",
            "candidates/search_page_candidates.jsonl",
            "candidates/agentic_candidates.jsonl",
            "logs/validation_results.jsonl",
            "camera.geojson",
        ]
        created = [rel for rel in broad_artifacts if (run_dir / rel).exists() and (run_dir / rel).stat().st_size > 0]
        if created:
            warn("Discovery/validation artifacts exist despite insufficient scope: " + ", ".join(created))



## Real Insufficient-Scope Validation

Runs the real CLI with a deliberately broad query and verifies the LLM stops discovery before search, validation, catalog, or map work.


In [ ]:
insufficient_scope_passed = None
insufficient_scope_zip_path = None
if RUN_INSUFFICIENT_SCOPE_VALIDATION:
    insufficient_scope_run_dir = pathlib.Path(ARTIFACT_ROOT) / f"insufficient_scope_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
    if insufficient_scope_run_dir.exists():
        shutil.rmtree(insufficient_scope_run_dir)
    insufficient_scope_run_dir.mkdir(parents=True, exist_ok=True)
    insufficient_cmd = [
        "webcam-discovery", "run-agentic", INSUFFICIENT_SCOPE_QUERY,
        "--output-dir", str(insufficient_scope_run_dir),
        "--max-search-queries", "5",
        "--max-search-results-per-query", "3",
        "--max-candidates", "10",
        "--max-streams", "2",
    ]
    if "--disable-ffprobe-validation" in help_text:
        insufficient_cmd.append("--disable-ffprobe-validation")
    if "--disable-validation" in help_text:
        insufficient_cmd.append("--disable-validation")
    if "--llm-provider" in help_text:
        insufficient_cmd += ["--llm-provider", PLANNER_PROVIDER]
    if "--llm-model" in help_text:
        insufficient_cmd += ["--llm-model", PLANNER_MODEL]

    insufficient_log = insufficient_scope_run_dir / "notebook_run_stdout_stderr.log"
    insufficient_rc, insufficient_elapsed = run_with_heartbeat(
        insufficient_cmd,
        cwd=repo_dir,
        log_path=insufficient_log,
        env=env,
        heartbeat_seconds=HEARTBEAT_SECONDS,
        max_runtime_minutes=20,
        recent_lines=HEARTBEAT_RECENT_LINES,
        write_status_json=HEARTBEAT_STATUS_JSON,
    )
    print("INSUFFICIENT_SCOPE_RUN_DIR:", insufficient_scope_run_dir)
    print("RETURN_CODE:", insufficient_rc, "ELAPSED_SECONDS:", insufficient_elapsed)

    insufficient_scope = load_json(insufficient_scope_run_dir / "logs" / "scope_inference.json") or {}
    insufficient_summary = load_json(insufficient_scope_run_dir / "logs" / "run_summary.json") or {}
    if insufficient_scope.get("has_sufficient_scope") is not False:
        raise RuntimeError("Expected insufficient scope from real LLM scope inference, got: " + json.dumps(insufficient_scope, ensure_ascii=False)[:2000])

    forbidden_after_scope = [
        "logs/search_queries.jsonl",
        "logs/search_results.jsonl",
        "candidates/search_page_candidates.jsonl",
        "candidates/agentic_candidates.jsonl",
        "logs/validation_results.jsonl",
        "logs/ffprobe_validation.jsonl",
        "logs/visual_stream_analysis.jsonl",
        "camera.geojson",
        "map.html",
    ]
    created = [rel for rel in forbidden_after_scope if (insufficient_scope_run_dir / rel).exists() and (insufficient_scope_run_dir / rel).stat().st_size > 0]
    if created:
        raise RuntimeError("Insufficient-scope run created discovery/validation/catalog artifacts: " + ", ".join(created))
    log_text = insufficient_log.read_text(encoding="utf-8", errors="replace") if insufficient_log.exists() else ""
    broad_markers = ["SearchAgent", "FeedDiscoveryAgent", "DeepDiscoveryAgent", "ValidationAgent:", "CatalogAgent", "MapAgent"]
    found_markers = [m for m in broad_markers if m in log_text]
    if found_markers:
        raise RuntimeError("Insufficient-scope subprocess log shows forbidden post-scope stages: " + ", ".join(found_markers))
    insufficient_scope_passed = True
    print("Insufficient-scope validation passed with real app artifacts.")

    insufficient_scope_zip_path = pathlib.Path(str(insufficient_scope_run_dir) + ".zip")
    if insufficient_scope_zip_path.exists():
        insufficient_scope_zip_path.unlink()
    with zipfile.ZipFile(insufficient_scope_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for p in insufficient_scope_run_dir.rglob("*"):
            if p.is_file():
                z.write(p, p.relative_to(insufficient_scope_run_dir.parent))
    print("Created insufficient-scope zip:", insufficient_scope_zip_path)
else:
    print("RUN_INSUFFICIENT_SCOPE_VALIDATION=False; skipped.")



## Optional Real Multi-Scope Validation


In [ ]:
multi_scope_passed = None
if RUN_MULTI_SCOPE_VALIDATION:
    multi_scope_run_dir = pathlib.Path(ARTIFACT_ROOT) / f"multi_scope_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
    multi_scope_run_dir.mkdir(parents=True, exist_ok=True)
    multi_cmd = [
        "webcam-discovery", "run-agentic", MULTI_SCOPE_QUERY,
        "--output-dir", str(multi_scope_run_dir),
        "--max-search-queries", "8",
        "--max-search-results-per-query", "4",
        "--max-candidates", "20",
        "--max-streams", "5",
        "--disable-ffprobe-validation",
        "--disable-validation",
    ]
    if "--llm-provider" in help_text:
        multi_cmd += ["--llm-provider", PLANNER_PROVIDER]
    if "--llm-model" in help_text:
        multi_cmd += ["--llm-model", PLANNER_MODEL]
    multi_log = multi_scope_run_dir / "notebook_run_stdout_stderr.log"
    multi_rc, multi_elapsed = run_with_heartbeat(multi_cmd, cwd=repo_dir, log_path=multi_log, env=env, heartbeat_seconds=HEARTBEAT_SECONDS, max_runtime_minutes=30)
    multi_scope = load_json(multi_scope_run_dir / "logs" / "scope_inference.json") or {}
    targets_text = json.dumps(multi_scope, ensure_ascii=False).lower()
    if "london" not in targets_text or "sydney" not in targets_text:
        raise RuntimeError("Multi-scope validation did not show both requested targets in scope artifacts: " + json.dumps(multi_scope, ensure_ascii=False)[:2000])
    multi_scope_passed = True
    print("Multi-scope validation passed:", multi_scope_run_dir)
else:
    print("RUN_MULTI_SCOPE_VALIDATION=False; skipped.")



## 11. Step 1 — Planner and query-generation audit

This catches whether the planner or query generator broadened the request beyond the query-derived target scope. Same-name leakage is detected using target-specific negative or ambiguous terms derived from the query, not California-specific lists.


In [ ]:
planner_files = [
    run_dir / "logs" / "search_plan.json",
    run_dir / "logs" / "planner_runs.jsonl",
    run_dir / "logs" / "search_queries.jsonl",
]
planner_rows = []
for p in planner_files:
    if p.suffix == ".json":
        obj = load_json(p)
        if obj:
            planner_rows.append({"artifact": str(p.relative_to(run_dir)), "text": json.dumps(obj, ensure_ascii=False)[:5000]})
    else:
        for r in load_jsonl(p):
            planner_rows.append({"artifact": str(p.relative_to(run_dir)), "text": json.dumps(r, ensure_ascii=False)[:5000]})

print("Planner/query rows:", len(planner_rows))
if planner_rows:
    pdf = pd.DataFrame(planner_rows)
    display(pdf.head(100))

leak_rows = []
for r in planner_rows:
    blob = r["text"].lower()
    hits = [t for t in TARGET_NEGATIVE_TERMS if t and t.lower() in blob]
    if hits:
        rr = dict(r)
        rr["negative_or_ambiguous_term_hits"] = ", ".join(hits)
        leak_rows.append(rr)

if leak_rows:
    warn(f"Planner/search query artifacts contain off-target or ambiguous same-name terms: {len(leak_rows)} row(s).")
    display(pd.DataFrame(leak_rows))
else:
    print("No query-derived off-target/ambiguous terms found in planner/query artifacts.")

# Positive-term coverage check: a plan with no query-derived target evidence is risky.
missing_target_evidence = []
for r in planner_rows:
    blob = r["text"].lower()
    if TARGET_POSITIVE_TERMS and not any(t in blob for t in TARGET_POSITIVE_TERMS):
        rr = dict(r)
        rr["issue"] = "no positive target term found in planner/query row"
        missing_target_evidence.append(rr)
if missing_target_evidence:
    warn(f"Some planner/query rows lack derived target evidence: {len(missing_target_evidence)} row(s).")
    display(pd.DataFrame(missing_target_evidence).head(100))


## 12. Step 2 — Search-result audit

This examines raw search results before feed extraction. It shows whether off-target locations entered through search results, before any validation or cataloging occurs.


In [ ]:
search_rows = load_jsonl(run_dir / "logs" / "search_results.jsonl")
for r in search_rows:
    r["domain"] = row_domain(r)
    blob = row_text(r)
    r["negative_term_hits"] = ", ".join([t for t in TARGET_NEGATIVE_TERMS if t.lower() in blob])
    r["positive_term_hits"] = ", ".join([t for t in TARGET_POSITIVE_TERMS + TARGET_REGION_TERMS if t.lower() in blob])

print("Search result rows:", len(search_rows))
if search_rows:
    sdf = pd.DataFrame(search_rows)
    keep = [c for c in ["domain", "query", "title", "url", "snippet", "positive_term_hits", "negative_term_hits"] if c in sdf.columns]
    display(sdf[keep].head(150) if keep else sdf.head(150))
    display(sdf.groupby("domain").size().reset_index(name="count").sort_values("count", ascending=False).head(50))

    off = sdf[sdf["negative_term_hits"].astype(str).str.len() > 0]
    if len(off):
        warn(f"Search results contain configured off-target terms: {len(off)} row(s).")
        display(off[keep].head(100) if keep else off.head(100))
else:
    warn("No search_results.jsonl found. If this is expected, confirm the app writes equivalent search-stage artifacts.")


## 13. Step 3 — Discovery funnel and cap audit

This is the most important place to inspect under-discovery or over-discovery. It flattens `discovery_funnel.json` and compares observed counts to configured limits where possible.


In [ ]:
funnel_path = run_dir / "logs" / "discovery_funnel.json"
funnel = load_json(funnel_path) or {}
flat = flatten("", funnel, {}) if funnel else {}
summary_df = pd.DataFrame([{"metric": k, "value": v} for k, v in sorted(flat.items())])
print("Funnel path:", funnel_path, "exists=", funnel_path.exists())
display(summary_df.head(300))

# Fuzzy cap audit: metrics names may vary, so match by keyword.
cap_specs = [
    ("max_feed_endpoints", MAX_FEED_ENDPOINTS, ["feed_endpoint", "feed.endpoints", "endpoints"]),
    ("max_feed_records", MAX_FEED_RECORDS, ["feed_record", "feed.records", "records"]),
    ("max_stream_candidates", MAX_STREAM_CANDIDATES, ["stream_candidates", "stream.candidates", "candidates"]),
    ("per_source_stream_cap", PER_SOURCE_STREAM_CAP, ["per_source", "source_stream"]),
    ("max_streams", MAX_STREAMS, ["validation.received", "streams_to_catalog", "hls streams", "max_streams"]),
    ("max_search_queries", MAX_SEARCH_QUERIES, ["search_queries", "queries"]),
]

cap_rows = []
for cap_name, cap_value, hints in cap_specs:
    for metric, value in flat.items():
        if not isinstance(value, (int, float)):
            continue
        m = metric.lower()
        if any(h.lower() in m for h in hints):
            cap_rows.append({
                "cap_name": cap_name,
                "cap_value": cap_value,
                "metric": metric,
                "observed_value": value,
                "over_cap": value > cap_value,
            })
cap_df = pd.DataFrame(cap_rows)
if not cap_df.empty:
    display(cap_df.sort_values(["over_cap", "cap_name", "observed_value"], ascending=[False, True, False]).head(200))
    violations = cap_df[cap_df["over_cap"]]
    if len(violations):
        warn(f"Potential cap violations detected in funnel metrics: {len(violations)}.")
else:
    warn("No cap-related metrics found in discovery_funnel.json. Add explicit per-stage counts to the app logs.")

# Specific under-discovery signal: many discovered pages/streams, few sent to validation.
before_keys = [k for k in flat if any(s in k.lower() for s in ["stream_candidates", "streams_found", "candidate", "resolved"])]
validation_keys = [k for k in flat if "validation" in k.lower() and any(s in k.lower() for s in ["received", "candidate", "stream"])]
print("Candidate-like metric keys:", before_keys[:50])
print("Validation-like metric keys:", validation_keys[:50])


## 14. Step 4 — Candidate artifacts, priority, and rejection audit

This identifies whether valid-looking streams are rejected before validation, deprioritized too aggressively, or lost by priority/cap selection.


In [ ]:
all_rows = load_all_jsonl(run_dir)
print("All JSONL rows loaded:", len(all_rows))

candidate_artifacts = [r for r in all_rows if any(k in r.get("_artifact_rel", "") for k in ["candidate", "priority", "rejected", "deprioritized"])]
print("Candidate-related rows:", len(candidate_artifacts))
if candidate_artifacts:
    cdf = pd.DataFrame(candidate_artifacts)
    keep = [c for c in ["_artifact_rel", "domain", "priority", "priority_reason", "sent_to_validation", "stage", "reason", "candidate_url", "stream_url", "url"] if c in cdf.columns]
    display(cdf[keep].head(200) if keep else cdf.head(200))
    if "_artifact_rel" in cdf.columns:
        display(cdf.groupby(["_artifact_rel", "domain"]).size().reset_index(name="count").sort_values("count", ascending=False).head(100))

# Check weak-evidence hard rejections.
weak_reasons = [
    "no source-page target evidence", "query-only target evidence", "missing camera/hls evidence",
    "untrusted cdn without strong lineage", "insufficient target evidence", "weak target evidence",
]
weak_pre_validation = []
for r in candidate_artifacts:
    reason = str(r.get("reason", "")).lower()
    stage = str(r.get("stage", "")).lower()
    if any(w in reason for w in weak_reasons) and stage not in {"validation", "robots", "policy", "malformed_url", "non_hls"}:
        weak_pre_validation.append(r)
if weak_pre_validation:
    warn(f"Weak-evidence pre-validation hard rejections detected: {len(weak_pre_validation)}.")
    display(pd.DataFrame(weak_pre_validation).head(100))
else:
    print("No weak-evidence pre-validation hard rejections detected in candidate artifacts.")

# Per-domain candidate cap audit.
streamish = [r for r in candidate_artifacts if ".m3u8" in row_text(r) or "playlist" in row_text(r)]
if streamish:
    sdf = pd.DataFrame(streamish)
    domain_counts = sdf.groupby("domain").size().reset_index(name="streamish_rows").sort_values("streamish_rows", ascending=False)
    domain_counts["over_PER_SOURCE_STREAM_CAP"] = domain_counts["streamish_rows"] > PER_SOURCE_STREAM_CAP
    display(domain_counts.head(100))
    if domain_counts["over_PER_SOURCE_STREAM_CAP"].any():
        warn("One or more domains have more streamish candidate rows than PER_SOURCE_STREAM_CAP. Confirm cap is applied at the intended stage.")


## 15. Step 5 — Query-derived target leakage audit across every artifact

This scans artifacts for off-target or ambiguous terms derived from the user query. If a bbox is present in the query-derived/manual target scope, GeoJSON coordinates are checked against that bbox. No bbox is defaulted by the notebook.


In [ ]:
leakage_rows = []
for r in all_rows:
    blob = row_text(r)
    neg_hits = [t for t in TARGET_NEGATIVE_TERMS if t and t.lower() in blob]
    if neg_hits:
        rr = {
            "artifact": r.get("_artifact_rel"),
            "line": r.get("_line"),
            "domain": r.get("domain"),
            "negative_or_ambiguous_term_hits": ", ".join(neg_hits),
            "url": row_url(r),
            "preview": blob[:500],
        }
        leakage_rows.append(rr)

print("Artifact rows with query-derived off-target/ambiguous terms:", len(leakage_rows))
if leakage_rows:
    warn(f"Potential target leakage detected in JSONL artifacts: {len(leakage_rows)} row(s).")
    display(pd.DataFrame(leakage_rows).head(200))
else:
    print("No query-derived off-target/ambiguous terms found in JSONL artifacts.")

# GeoJSON boundary audit only when target scope includes a bbox.
geojson_path = run_dir / "camera.geojson"
features = []
if geojson_path.exists():
    obj = load_json(geojson_path) or {}
    features = obj.get("features", []) if isinstance(obj, dict) else []

geo_rows = []
for i, f in enumerate(features):
    geom = f.get("geometry") or {}
    coords = geom.get("coordinates") or []
    lon = coords[0] if len(coords) > 0 else None
    lat = coords[1] if len(coords) > 1 else None
    props = f.get("properties") or {}
    props_text = json.dumps(props, ensure_ascii=False).lower()
    geo_rows.append({
        "idx": i,
        "lat": lat,
        "lon": lon,
        "inside_query_bbox": in_bbox(lon, lat, TARGET_BBOX),
        "positive_scope_evidence": ", ".join([t for t in TARGET_POSITIVE_TERMS if t in props_text][:10]),
        "negative_scope_hits": ", ".join([t for t in TARGET_NEGATIVE_TERMS if t in props_text][:10]),
        "name": props.get("name") or props.get("title") or props.get("label"),
        "url": props.get("url") or props.get("stream_url"),
        "properties_preview": json.dumps(props, ensure_ascii=False)[:500],
    })

print("GeoJSON features:", len(geo_rows))
if geo_rows:
    gdf = pd.DataFrame(geo_rows)
    display(gdf.head(200))
    neg_geo = gdf[gdf["negative_scope_hits"].astype(str).str.len() > 0]
    if len(neg_geo):
        warn(f"GeoJSON properties include query-derived off-target/ambiguous terms: {len(neg_geo)} feature(s).")
        display(neg_geo)
    if TARGET_BBOX is not None:
        outside = gdf[gdf["inside_query_bbox"] == False]
        if len(outside):
            warn(f"GeoJSON contains features outside query-derived bbox: {len(outside)}.")
            display(outside)
else:
    print("No GeoJSON features found or camera.geojson missing.")


## 16. Step 6 — LLM URL and metadata extraction audit

This audit is not tied to WZMedia, Caltrans, or any other source. It gathers real URLs from actual run artifacts, then uses LLM extraction to identify district/region, route, direction, road/street, locality, agency/source, and place hints. There is no heuristic metadata parser in this notebook.


In [ ]:
# Gather actual URLs from artifacts only. No synthetic/example stream URLs are added.
# Basic regex is used only to collect URL strings from real artifacts; metadata extraction below is LLM-only.
artifact_url_rows = []
for r in all_rows:
    blob = row_text(r)
    urls = set(re.findall(r"""https?://[^\s"'<>]+""", blob))
    direct = row_url(r)
    if direct:
        urls.add(direct)
    for url in urls:
        if not url or len(url) > 2000:
            continue
        if any(marker in url.lower() for marker in ["m3u8", "playlist", "stream", "camera", "cam", "cctv", "video"]):
            artifact_url_rows.append({
                "url": url,
                "domain": urlparse(url).netloc.lower(),
                "artifact": r.get("_artifact_rel"),
                "line": r.get("_line"),
                "context_preview": blob[:1200],
            })

print("Actual artifact URL rows prepared for LLM metadata extraction:", len(artifact_url_rows))

llm_url_rows = []
if artifact_url_rows:
    if not ENABLE_LLM_URL_METADATA_AUDIT or not USE_OLLAMA_CLOUD:
        raise RuntimeError(
            "Real camera-like artifact URLs were found, but LLM URL metadata extraction is disabled. "
            "Heuristic URL metadata parsing is intentionally disabled."
        )

    sample = (
        pd.DataFrame(artifact_url_rows)
        .drop_duplicates(subset=["url"])
        .head(MAX_LLM_URL_METADATA_ROWS)
        .to_dict("records")
    )
    print(f"Running LLM metadata extraction on {len(sample)} actual URL(s)...")
    for item in sample:
        try:
            llm_meta = llm_extract_url_metadata(item["url"], item.get("context_preview", ""), TARGET_SCOPE)
            llm_meta["url"] = item["url"]
            llm_meta["domain"] = item.get("domain")
            llm_meta["artifact"] = item.get("artifact")
            llm_meta["line"] = item.get("line")
            llm_url_rows.append(llm_meta)
        except Exception as exc:
            llm_url_rows.append({
                "url": item["url"],
                "domain": item.get("domain"),
                "artifact": item.get("artifact"),
                "line": item.get("line"),
                "llm_error": str(exc),
            })
else:
    warn("No actual camera/stream-like URLs found in artifacts for LLM metadata extraction audit.")

if llm_url_rows:
    llm_url_df = pd.DataFrame(llm_url_rows)
    display(llm_url_df.head(MAX_LLM_URL_METADATA_ROWS))
    if "llm_error" in llm_url_df.columns and llm_url_df["llm_error"].fillna("").astype(str).str.len().gt(0).any():
        warn("One or more LLM URL metadata extraction calls failed. These rows should be reviewed before cataloging.")
    if "ambiguous" in llm_url_df.columns and llm_url_df["ambiguous"].astype(str).str.lower().isin(["true", "1"]).any():
        warn("LLM metadata extraction marked one or more URLs as ambiguous. These should not be auto-cataloged without stronger target/location evidence.")
else:
    print("LLM URL metadata audit produced no rows.")


## 17. Step 7 — Validation and stream-status audit

This checks HTTP probe, ffprobe, and visual-analysis artifacts. It should show whether streams are live, dead, unknown, restricted, or visually questionable.


In [ ]:
validation_paths = [
    run_dir / "logs" / "validation_results.jsonl",
    run_dir / "logs" / "ffprobe_validation.jsonl",
    run_dir / "logs" / "visual_stream_analysis.jsonl",
]
validation_rows = []
for p in validation_paths:
    for r in load_jsonl(p):
        r["artifact"] = str(p.relative_to(run_dir))
        r["domain"] = row_domain(r)
        r["url_norm"] = row_url(r)
        validation_rows.append(r)

print("Validation rows:", len(validation_rows))
if validation_rows:
    vdf = pd.DataFrame(validation_rows)
    keep = [c for c in ["artifact", "domain", "stream_status", "status", "substatus", "frames", "brightness", "entropy", "diff_max", "fail_reason", "detail", "url_norm"] if c in vdf.columns]
    display(vdf[keep].head(300) if keep else vdf.head(300))
    for col in ["stream_status", "status", "substatus", "fail_reason"]:
        if col in vdf.columns:
            display(vdf.groupby(["artifact", col]).size().reset_index(name="count").sort_values("count", ascending=False).head(100))
else:
    warn("No validation artifacts found. Check whether validation ran and wrote logs/validation_results.jsonl or equivalent.")


## 18. Step 8 — Catalog, dedup, unknown-location, and map audit

This targets the previous failure where validated streams without coordinates collapsed into one `unknown-unknown` record and no GeoJSON features were exported.


In [ ]:
geojson_path = run_dir / "camera.geojson"
cameras_md_path = run_dir / "cameras.md"
map_path = run_dir / "map.html"
unknown_paths = [
    run_dir / "candidates" / "needs_review_location_unknown.jsonl",
    run_dir / "logs" / "needs_review_location_unknown.jsonl",
]
unknown_rows = []
for p in unknown_paths:
    unknown_rows.extend(load_jsonl(p))

print("camera.geojson exists:", geojson_path.exists(), "bytes:", geojson_path.stat().st_size if geojson_path.exists() else 0)
print("cameras.md exists:", cameras_md_path.exists(), "bytes:", cameras_md_path.stat().st_size if cameras_md_path.exists() else 0)
print("map.html exists:", map_path.exists(), "bytes:", map_path.stat().st_size if map_path.exists() else 0)
print("unknown-location review rows:", len(unknown_rows))

features = []
if geojson_path.exists():
    obj = load_json(geojson_path) or {}
    if isinstance(obj, dict):
        features = obj.get("features", [])
print("GeoJSON feature count:", len(features))

if unknown_rows:
    udf = pd.DataFrame(unknown_rows)
    for c in ["candidate_url", "stream_url", "url"]:
        if c in udf.columns:
            break
    display(udf.head(200))

# Detect unknown-unknown collapse in logs and catalog artifacts.
collapse_hits = []
for p in list((run_dir / "logs").rglob("*")) if (run_dir / "logs").exists() else []:
    if p.is_file() and p.suffix in {".log", ".txt", ".json", ".jsonl"}:
        text = p.read_text(encoding="utf-8", errors="replace").lower()[:2_000_000]
        if "unknown-unknown" in text or "dropped" in text and "dedup" in text:
            collapse_hits.append({"artifact": str(p.relative_to(run_dir)), "mentions_unknown_unknown": "unknown-unknown" in text, "mentions_dedup_drop": "dedup" in text and "dropped" in text})
# Include run stdout.
run_log = run_dir / "notebook_run_stdout_stderr.log"
if run_log.exists():
    text = run_log.read_text(encoding="utf-8", errors="replace").lower()
    if "unknown-unknown" in text or ("dedup" in text and "dropped" in text):
        collapse_hits.append({"artifact": str(run_log.relative_to(run_dir)), "mentions_unknown_unknown": "unknown-unknown" in text, "mentions_dedup_drop": "dedup" in text and "dropped" in text})

if collapse_hits:
    warn("Catalog/dedup logs suggest unknown-location collapse or dedup drops. Review table below.")
    display(pd.DataFrame(collapse_hits))

if len(features) == 0 and len(unknown_rows) > 0:
    warn("Validated/unknown-location rows exist but GeoJSON has zero features. Catalog should preserve reviewable records and avoid collapsing unknown locations.")


## 19. Step 9 — SSL, HTTP, and source-error audit

This scans the subprocess log and app logs for SSL failures, protocol errors, 403/404 patterns, and whether insecure fallback was used.


In [ ]:
log_texts = []
for p in [run_dir / "notebook_run_stdout_stderr.log"] + list((run_dir / "logs").rglob("*")) if (run_dir / "logs").exists() else [run_dir / "notebook_run_stdout_stderr.log"]:
    if pathlib.Path(p).is_file():
        try:
            log_texts.append((pathlib.Path(p).relative_to(run_dir), pathlib.Path(p).read_text(encoding="utf-8", errors="replace")))
        except Exception:
            pass

patterns = {
    "ssl_cert_verify_failed": r"CERTIFICATE_VERIFY_FAILED|unable to get local issuer certificate|SSL certificate verification failed",
    "insecure_fallback_used": r"fallback succeeded|retrying.*without certificate verification|verify=False",
    "http_403": r"\b403\b|Forbidden",
    "http_404": r"\b404\b|Not Found",
    "remote_protocol": r"RemoteProtocolError|Server disconnected",
    "timeout": r"timeout|timed out",
}
err_rows = []
for rel, text in log_texts:
    for name, pat in patterns.items():
        hits = re.findall(pat, text, flags=re.I)
        if hits:
            err_rows.append({"artifact": str(rel), "pattern": name, "count": len(hits)})

if err_rows:
    edf = pd.DataFrame(err_rows).sort_values(["pattern", "count"], ascending=[True, False])
    display(edf)
    if any(r["pattern"] == "ssl_cert_verify_failed" for r in err_rows):
        warn("SSL certificate verification failures occurred. Confirm WCD_CA_BUNDLE is used by the app HTTP client helper.")
else:
    print("No configured SSL/HTTP error patterns found in logs.")


## 20. Step 10 — Consolidated diagnostics and recommended code fixes

This cell summarizes the notebook findings into a markdown file you can attach to a Codex task or review with ChatGPT.


In [ ]:
recommendations = []

if leakage_rows:
    recommendations.append("Add or strengthen query-derived target-scope enforcement: separate user-target geocoding from candidate geocoding, require administrative/country/place agreement where available, and send off-target same-name matches to rejection/review before validation/catalog.")
if 'cap_df' in globals() and not cap_df.empty and cap_df.get("over_cap", pd.Series(dtype=bool)).any():
    recommendations.append("Clarify and enforce caps at each stage: search results, feed endpoints, feed records, stream candidates, validation candidates, per-source streams, and cataloged streams. Emit explicit counts before/after every cap so parameter overruns are explainable.")
if 'udf' in globals() and len(udf) > 0:
    recommendations.append("Implement source-agnostic LLM URL metadata extraction: extract district/region, route, direction, road/street, locality, and agency/source hints from URL/path/query/source context with evidence and confidence. Preserve extracted metadata through validation/catalog.")
if 'llm_url_df' in globals() and not llm_url_df.empty:
    recommendations.append("Use LLM URL metadata extraction with evidence/confidence fields so ambiguous same-name places are not silently accepted.")
if 'features' in globals() and len(features) == 0 and 'unknown_rows' in globals() and len(unknown_rows) > 0:
    recommendations.append("Fix catalog/dedup so location-unknown live streams are not collapsed into a single unknown-unknown record; use stable URL/source fingerprints for dedup and preserve review artifacts.")
if any("SSL" in w or "ssl" in w for w in warnings_accumulator):
    recommendations.append("Ensure all public discovery HTTP clients use a centralized certifi/trust_env-enabled client helper and only use insecure SSL fallback when explicitly enabled.")
if not TARGET_POSITIVE_TERMS:
    recommendations.append("Require the app to reject or ask for clarification when the user query does not provide a geocodable place, landmark, IP/location indicator, or other target scope.")
if not recommendations:
    recommendations.append("No major notebook-detected issues. Review tables manually for source-specific anomalies and quality concerns.")

report = {
    "query": QUERY,
    "target_scope": TARGET_SCOPE,
    "cli_validation_flags": {
        "DISABLE_FFPROBE_VALIDATION": DISABLE_FFPROBE_VALIDATION,
        "DISABLE_VALIDATION": DISABLE_VALIDATION,
    },
    "ignore_sources_md": IGNORE_SOURCES_MD,
    "scope_inference": scope_inference if 'scope_inference' in globals() else {},
    "scope_summary": scope_summary if 'scope_summary' in globals() else {},
    "search_scope_gate_counts": search_scope_counts if 'search_scope_counts' in globals() else {},
    "stream_scope_gate_counts": stream_scope_counts if 'stream_scope_counts' in globals() else {},
    "insufficient_scope_validation_passed": insufficient_scope_passed if 'insufficient_scope_passed' in globals() else None,
    "validation_skipped": (run_summary.get('validation') or {}).get('validation_skipped') if 'run_summary' in globals() else None,
    "ffprobe_validation_enabled": (run_summary.get('validation') or {}).get('ffprobe_validation_enabled') if 'run_summary' in globals() else None,
    "unvalidated_candidates_written": (run_dir / 'candidates' / 'unvalidated_stream_candidates.jsonl').exists() if 'run_dir' in globals() else False,
    "limit_preset": LIMIT_PRESET,
    "limits": limits,
    "run_dir": str(run_dir),
    "warnings": warnings_accumulator,
    "recommendations": recommendations,
}

report_path = run_dir / "debug_diagnostics_summary.md"
with report_path.open("w", encoding="utf-8") as f:
    f.write(f"# Webcam Discovery Debug Diagnostics\n\n")
    f.write(f"- Query: `{QUERY}`\n")
    f.write(f"- Target scope mode: `{TARGET_SCOPE.get('mode', TARGET_SCOPE_MODE)}`\n")
    f.write(f"- Limit preset: `{LIMIT_PRESET}`\n")
    f.write(f"- Run directory: `{run_dir}`\n")
    f.write(f"- DISABLE_FFPROBE_VALIDATION: `{DISABLE_FFPROBE_VALIDATION}`\n")
    f.write(f"- DISABLE_VALIDATION: `{DISABLE_VALIDATION}`\n")
    f.write(f"- IGNORE_SOURCES_MD: `{IGNORE_SOURCES_MD}`\n")
    f.write(f"- Insufficient-scope validation passed: `{insufficient_scope_passed if 'insufficient_scope_passed' in globals() else None}`\n")
    f.write(f"- Validation skipped: `{(run_summary.get('validation') or {}).get('validation_skipped') if 'run_summary' in globals() else None}`\n")
    f.write(f"- ffprobe validation enabled: `{(run_summary.get('validation') or {}).get('ffprobe_validation_enabled') if 'run_summary' in globals() else None}`\n")
    f.write(f"- Unvalidated candidates artifact exists: `{(run_dir / 'candidates' / 'unvalidated_stream_candidates.jsonl').exists()}`\n\n")
    f.write("## App LLM scope inference\n\n")
    f.write("```json\n")
    f.write(json.dumps(scope_inference if 'scope_inference' in globals() else {}, indent=2, ensure_ascii=False))
    f.write("\n```\n\n")
    f.write("## Search-result scope gate counts\n\n")
    f.write(json.dumps(search_scope_counts if 'search_scope_counts' in globals() else {}, indent=2, ensure_ascii=False) + "\n\n")
    f.write("## Stream-candidate scope gate counts\n\n")
    f.write(json.dumps(stream_scope_counts if 'stream_scope_counts' in globals() else {}, indent=2, ensure_ascii=False) + "\n\n")
    f.write("## Derived target scope\n\n")
    f.write("```json\n")
    f.write(json.dumps(TARGET_SCOPE, indent=2, ensure_ascii=False))
    f.write("\n```\n\n")
    f.write("## Warnings\n\n")
    for w in warnings_accumulator or ["No accumulated warnings."]:
        f.write(f"- {w}\n")
    f.write("\n## Recommendations\n\n")
    for r in recommendations:
        f.write(f"- {r}\n")
    f.write("\n## Effective limits\n\n")
    for k, v in limits.items():
        f.write(f"- {k}: {v}\n")

print(report_path.read_text(encoding="utf-8"))

if ASSERT_ON_WARNINGS and warnings_accumulator:
    raise AssertionError("Notebook diagnostics found warnings. See debug_diagnostics_summary.md.")


## 21. Package debug artifacts

The zip includes the raw stdout/stderr log, all application artifacts, audit CSVs/JSONL if the app wrote them, and the diagnostics summary.


In [ ]:
zip_path = pathlib.Path(str(run_dir) + ".zip")
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in run_dir.rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(run_dir.parent))
print("Created:", zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as exc:
    print("Download helper unavailable outside Colab:", exc)


## Review checklist

Use this checklist after the run:

1. The query has a geocodable target scope. If not, the app should reject it or ask for clarification instead of running a broad search.
2. Planner/search queries remain inside the query-derived target scope.
3. Search results do not include same-name off-target places unless clearly rejected later.
4. Discovery funnel counts explain every major drop from pages → feeds → stream candidates → validation → catalog.
5. Configured limits are respected at the stage they claim to control, or logs clearly explain why a larger intermediate count exists.
6. LLM URL metadata extraction produces usable district/region, route, direction, road/street, locality, and agency/source hints when those clues exist in actual URLs or source context.
7. LLM metadata extraction handles ambiguous URLs without hardcoding any agency, vendor, state, or country.
8. Live streams without coordinates are preserved for review and do not collapse into one `unknown-unknown` catalog entry.
9. GeoJSON features stay inside a query-derived/manual bbox only when such a bbox exists; the notebook never defaults to California or any other place.
10. Heartbeat output makes current stage, findings, cap usage, and location leakage visible while the subprocess is still running.
11. SSL failures are visible and not silently masked by global `verify=False`.
